---
title: Model Sky
subtitle: Follow a star to a hidden world
execute:
  enabled: false
code-links:
  - text: Notebook source on GitHub
    icon: github
    href: https://github.com/NVIDIA/cuPhoton/blob/main/examples/model-sky/model-sky.ipynb
  - text: Example code on GitHub
    icon: folder
    href: https://github.com/NVIDIA/cuPhoton/tree/main/examples/model-sky
---

A planet usually disappears into its star's glare. We can still notice it
when it crosses in front of the star and hides a little light. That temporary
dimming is a **<a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>**. Measuring it requires more than finding a dip in a
plot: we need to know which star dimmed, whether the images line up, and
whether the apparent change could come from the camera or atmosphere.

This notebook follows one cumulative experiment. First a satellite spoils
part of an <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a>. We identify those pixels, align the images and subtract
the steady field. Next, the paired bright and dark footprint of a moving star
reveals where it went. Finally, we measure that same star's light across a
simulated transit and ask what the evidence can actually establish.

**Model Sky builds every sky pixel and instrument input from declared
models.** The stars, satellite trail, observing times, motion and transit are
simulations. Their known answers stay in a separate file until measurement is
complete. A controlled sky lets us follow cause and effect: we know what was
put into the images, then ask whether measurements recover it. These are
teaching observations of a fictional field, not an observed planet discovery.

The explanations unpack the units, assumptions and interpretation behind each
measurement. The model is deliberately simpler than a real sky or camera; a
successful recovery establishes behavior within those assumptions, not accuracy
for every astronomical observation.

You can follow the main text without knowing astronomy or calculus. Defined
terms have dotted underlines: hover or focus for a short definition, or follow
the link to the glossary. On touch screens, tap a term. Each collapsed
**Science Explanation** supplies the assumptions and equations behind the
step. Expand it when you want to inspect how the measurement works.

[NVIDIA cuPhoton](https://github.com/NVIDIA/cuPhoton) supplies GPU image
<a class="model-sky-glossary-term" href="#model-sky-glossary-reprojection">reprojection</a> (**xRep**), image matching and subtraction (**xPois**), and source
measurements (**xFit**). The notebook keeps those calls visible. Example
helpers manage the sequence and displays. **xScan** is not used: this experiment
includes no trained classifier or classifier-performance claim. The [NVIDIA imaging walkthrough](https://developer.nvidia.com/blog/faster-scientific-image-analysis-with-nvidia-cuphoton/)
provides further context; its older API examples do not replace the calls
in this example. The [quickstarts](https://github.com/NVIDIA/cuPhoton/blob/main/docs/quickstarts.md)
and [other cuPhoton demos](https://nvidia.github.io/cuPhoton/) cover more uses.

The illustrations below are schematics. Actual pixels and measured results follow.

<div class="model-sky-story-grid" role="group" aria-label="Our journey from a satellite trail to a hidden world" style="display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,240px),1fr));gap:16px">
<figure class="model-sky-story-card" style="margin:0;min-width:0"><img src="assets/story-field.svg" alt="A satellite trail crosses a star field; a mask covers the trail." width="300" height="360" style="display:block;width:100%;height:auto" /><figcaption style="padding:18px 16px"><strong>Spot the intruder</strong><p>That bright line is close to home: a satellite moved while the camera watched. We’ll mark the affected pixels so its light doesn’t spoil our star measurements.</p></figcaption></figure>
<figure class="model-sky-story-card" style="margin:0;min-width:0"><img src="assets/story-compare.svg" alt="Reference and new images reveal blue negative and red positive residuals after subtraction." width="300" height="360" style="display:block;width:100%;height:auto" /><figcaption style="padding:18px 16px"><strong>Make change stand out</strong><p>Line up two images and match the stars’ shapes. Subtract a <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> view: steady light largely cancels, while changing objects leave clues we can measure.</p></figcaption></figure>
<figure class="model-sky-story-card" style="margin:0;min-width:0"><img src="assets/story-motion.svg" alt="The same star shifts between two epochs against fixed neighbors." width="300" height="360" style="display:block;width:100%;height:auto" /><figcaption style="padding:18px 16px"><strong>Follow a star</strong><p>Our simulated star has shifted since the earlier image. Measure its position against neighboring stars, turn pixels into an angle, and work out how fast it moves across the sky.</p></figcaption></figure>
<figure class="model-sky-story-card" style="margin:0;min-width:0"><img src="assets/story-transit.svg" alt="An illustrative planet crosses a star, producing a dip in its unresolved light curve." width="300" height="360" style="display:block;width:100%;height:auto" /><figcaption style="padding:18px 16px"><strong>Find a hidden world</strong><p>Now follow that star’s light. A small, temporary dip hints at a passing planet. We’ll fit the simulated <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>, check nearby stars, and ask what else could explain the signal.</p></figcaption></figure>
</div>

<a id="model-sky-what-you-will-make"></a>

## What you will make

You will produce an aligned image pair, a defensible trail <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a>, a stellar
<a class="model-sky-glossary-term" href="#model-sky-glossary-displacement">displacement</a> with <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a>, and a <a class="model-sky-glossary-term" href="#model-sky-glossary-light-curve">light curve</a> with a fitted <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>. A
light curve is brightness plotted against time. The final question is
whether the measurements support the interpretation, including the possibility
that the interpretation is wrong.

Prepare the inputs with `python model_sky.py --output data`, then run the cells
in order. The notebook reads `data/` and writes intermediates under
`results/`. The baseline and <a class="model-sky-glossary-term" href="#model-sky-glossary-blend">blend</a> switches start disabled in the source.
The runner (`python run.py --output model-sky-output`) enables both in an
execution copy and keeps the complete results. Choose a fresh output directory
for each run. No classifier model or xScan inference is included.

<a id="model-sky-know-the-arrays-before-measuring-them"></a>

### Know the arrays before measuring them

A **<a class="model-sky-glossary-term" href="#model-sky-glossary-fits">FITS</a>** file is astronomy's container for arrays and descriptive metadata.
These modeled <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposures</a> have three named image planes with matching shapes:

| Plane | Meaning | What consumes it |
| --- | --- | --- |
| `SCI` | <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">Calibrated</a> light assigned to each pixel, in <a class="model-sky-glossary-term" href="#model-sky-glossary-njy">nJy</a> | <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">Registration</a>, subtraction and source fits |
| `VAR` | Estimated per-pixel <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a>, in nJy² | Weights and <a class="model-sky-glossary-term" href="#model-sky-glossary-conditional">conditional error</a> estimates |
| `MASK` | Integer quality flags; zero means usable here | Pixel exclusion and retained-light checks |
| `SCI` header | Coordinates, units and observing metadata | <a class="model-sky-glossary-term" href="#model-sky-glossary-wcs">WCS</a> mapping and plot labels |

A **nanojansky**, written **nJy**, is a spectral flux-density unit: one nJy is
10⁻³⁵ W m⁻² Hz⁻¹. Here it represents the photon-weighted brightness in the
<a class="model-sky-glossary-term" href="#model-sky-glossary-bandpass">modeled band</a>, assigned to each pixel. Fitting a <a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalized</a> star shape combines
pixels into a stellar <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a> in nJy. A variance in nJy² describes the spread of
repeated noisy measurements; its square root is a one-sigma error in nJy.
Negative background-subtracted pixels are possible even though <a class="model-sky-glossary-term" href="#model-sky-glossary-photon">photon</a> counts
are nonnegative.

The next cell checks the GPU and reports the installed cuPhoton version and
angular <a class="model-sky-glossary-term" href="#model-sky-glossary-angular-scale">pixel scale</a>. The manifest names measurement inputs and declared
assumptions; loading it does not read the separate injection-truth file.


<details class="model-sky-science"><summary>Science Explanation</summary>

**A noiseless model can still produce noisy references.** Preparation starts
from a fixed analytic field, with no measured <a class="model-sky-glossary-term" href="#model-sky-glossary-coadd">coadd</a> or catalog underneath it.
That latent field specifies the expected source light before each exposure's
<a class="model-sky-glossary-term" href="#model-sky-glossary-seeing">seeing</a>, pointing offset and transparency. It contains no random detector
<a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a>. New photon and sky/read fluctuations are drawn independently for every
exposure; copying one noisy image would instead copy its fluctuations.

For a modeled pixel M in nJy, let M₊ = max(M, 0), a be the conversion in nJy
per detected <a class="model-sky-glossary-term" href="#model-sky-glossary-electron">electron</a>, and V<sub>b</sub> the sky/read variance in nJy²:

<p class="model-sky-equation">Var(pixel) = a M₊ + V<sub>b</sub>.</p>

Photon arrivals follow <a class="model-sky-glossary-term" href="#model-sky-glossary-poisson">Poisson statistics</a>: an electron count's variance equals
its mean. The source term therefore becomes a M₊ after conversion to nJy.
The high sky count and detector <a class="model-sky-glossary-term" href="#model-sky-glossary-read-noise">read noise</a> are represented by <a class="model-sky-glossary-term" href="#model-sky-glossary-gaussian">Gaussian</a> draws.
The mean sky is subtracted, while its noise remains. Averaging N independent
equal exposures divides the variance by N. Both prepared references represent
16 such exposures; individual science frames use their own independent draws.

The <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a>'s finite noise matters when subtracting it and when estimating
positions or a <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> from it. Reusing a reference can couple later measurements,
even though the latent model was noiseless. Qualification changes exposure
noise conditional on this one field; it does not cover uncertain source
populations, unmodeled atmospheric behavior or every possible detector.
[The FITS standard](https://fits.gsfc.nasa.gov/fits_standard.html) documents
the file format independently of these modeling choices.

</details>

In [ ]:
import os
from pathlib import Path
import cupy as cp
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from bokeh.io import output_notebook
from notebook_style import show_dark
from bokeh.resources import INLINE
from IPython.display import Markdown, display
import cuphoton
import pipeline
import views
import workshop

# Input data and generated results have separate directories.
DATA = Path(os.environ.get("CUPHOTON_MODEL_SKY_DATA", "data")).resolve()
OUTPUT = Path(os.environ.get("CUPHOTON_MODEL_SKY_OUTPUT", "results"))
OUTPUT.mkdir(parents=True, exist_ok=True)
# Paths and assumptions only; injection truth is read later.
root, manifest = pipeline.load_bundle(DATA)
output_notebook(resources=INLINE, hide_banner=True)
pipeline.gpu()
# Angular size of one pixel converts displacement into sky motion.
scale = float(manifest["pixel_scale_arcsec"])
assumptions = manifest["assumptions"]
print("cuPhoton", cuphoton.__version__, "· device", cp.cuda.Device().id)
print(f"Pixel scale: {scale:.5f} arcsec/pixel")

The <a class="model-sky-glossary-term" href="#model-sky-glossary-angular-scale">pixel scale</a> converts an image <a class="model-sky-glossary-term" href="#model-sky-glossary-displacement">displacement</a> into an angle on the sky. The
version and device line identify what actually ran. A working GPU does not
establish scientific correctness; the image and control checks assess that
within their stated scope.

<a id="model-sky-1-a-satellite-crosses-the-field"></a>

## 1 · A satellite crosses the field

Start with a prepared Model Sky <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a> containing a modeled satellite trail.
The long feature crosses unrelated stars because the teaching model adds
light along a path during the exposure. A real nearby satellite can produce
such a streak by moving while reflecting sunlight into the telescope. Here
its shape and brightness are chosen explicitly; no satellite image is used.

The next cell reads the same noisy trail exposure that the measurement steps
will analyze, including the fictional target star. Pan, zoom and hover to
inspect native pixels. The display stretches brightness to show faint
structure; its colors are a viewing choice, while hover values retain units.
The sky-coordinate labels describe our declared grid, not a catalog match.


<details class="model-sky-science"><summary>Science Explanation</summary>

`model_sky.py` creates a 480 × 480 pixel field of isolated fictional stars. All
stars start with the same <a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalized</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-gaussian">Gaussian</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> of width <a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">σ</a> = 1.5 pixels;
the generator adds a common <a class="model-sky-glossary-term" href="#model-sky-glossary-seeing">seeing</a> blur per exposure. A Gaussian is a smooth
bell-shaped profile. Its width is a model choice, not a measured telescope PSF.

The modeled unobstructed telescope has a 4 m diameter. Its total <a class="model-sky-glossary-term" href="#model-sky-glossary-photon">photon</a>
response is 0.4 from 560 to 700 nm and zero outside: a simple <a class="model-sky-glossary-term" href="#model-sky-glossary-top-hat">top-hat</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-bandpass">bandpass</a>.
Each exposure lasts 30 seconds. The <a class="model-sky-glossary-term" href="#model-sky-glossary-background">sky level</a> is 20.5 AB mag/arcsec², the
<a class="model-sky-glossary-term" href="#model-sky-glossary-angular-scale">pixel scale</a> is 0.2 arcsec/pixel, and <a class="model-sky-glossary-term" href="#model-sky-glossary-read-noise">read noise</a> is 5 <a class="model-sky-glossary-term" href="#model-sky-glossary-electron">electrons</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-rms">RMS</a> per pixel
per exposure.
These assumptions set the brightness-to-electron conversion and <a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a>, not
just the labels on a plot. Scintillation, <a class="model-sky-glossary-term" href="#model-sky-glossary-saturation">saturation</a> and color-dependent
response changes are omitted.

The trail uses an analytic double-lobed cross section with a declared width
of 2.8 <a class="model-sky-glossary-term" href="#model-sky-glossary-arcsecond">arcsec</a>. It is a teaching approximation, not an optical ray trace or a
prediction for a particular satellite. A <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> can exclude contaminated pixels
from a fit, but cannot reconstruct the astronomical photons underneath them.
Masking trades information for a more trustworthy measurement.

[Tyson et al. (2020)](https://doi.org/10.3847/1538-3881/abba3e) discuss real
satellite-trail effects. The Model Sky inputs and their assumptions are recorded
in `data/provenance/` and the bundled source list.

</details>

In [ ]:
# A modeled noisy exposure; no observed image is loaded.
opening = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["trail"]))
show_dark(views.observed_image(
    opening["science"], title="Model Sky field · modeled satellite trail",
    eyebrow="Model Sky imagery",
    pixel_scale=scale, wcs=WCS(opening["header"]), unit="nJy"))

The opening picture and the following fits use the same modeled <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a>.
Its known origin makes it possible to test a measurement, but the measurement
still has to handle noisy pixels, changing blur and the trail. The generator's
hidden target and <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a> answers stay out of the image-analysis code.

<a id="model-sky-put-the-same-sky-on-the-same-pixels"></a>

### Put the same sky on the same pixels

Keep the same Model Sky <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a> and load its comparison frame. The
**<a class="model-sky-glossary-term" href="#model-sky-glossary-science-image">science image</a>** is the exposure whose change we want to measure. The **<a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a>
image** supplies the comparison sky. Those names describe roles, not different
file formats.

The <a class="model-sky-glossary-term" href="#model-sky-glossary-wcs">World Coordinate System</a> (**WCS**) maps pixel locations to directions on
the sky. Pointing changes mean a stationary star can occupy different array
indices in two exposures. **<a class="model-sky-glossary-term" href="#model-sky-glossary-reprojection">Reprojection</a>** resamples an image onto another sky coordinate grid. xRep receives science, <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a> and <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> planes plus a
reference-to-science coordinate map, and returns all three planes on the
reference grid. `aligned` holds reprojected science, not a <a class="model-sky-glossary-term" href="#model-sky-glossary-difference">difference image</a>.


<details class="model-sky-science"><summary>Science Explanation</summary>

For output location (x, y), reprojection finds the corresponding input
location and interpolates nearby input pixels. <a class="model-sky-glossary-term" href="#model-sky-glossary-interpolation">Interpolation</a> estimates values
between samples; it cannot restore detail that was never resolved. Neighboring
output pixels reuse input pixels, so their <a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a> becomes correlated. A variance
image records individual <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainties</a>, not the full <a class="model-sky-glossary-term" href="#model-sky-glossary-covariance">covariance</a> between pixels.

The coordinate grid is declared exactly as a TAN, or tangent-plane,
projection centered at right ascension 210° and declination −10°. These are
sky directions, analogous to longitude and latitude. The reference grid's
scale is 0.2 arcsec/pixel, with increasing x pointing toward decreasing right
ascension. The WCS is an input to the model, not a solution fitted to observed
stars, and no <a class="model-sky-glossary-term" href="#model-sky-glossary-empirical">empirical</a> sky-calibration accuracy is claimed.

Knowing the model grid does not make sampled images perfectly aligned.
<a class="model-sky-glossary-term" href="#model-sky-glossary-sampling">Subpixel</a> shifts and interpolation affect pixel values; noise affects estimated
centers. Subsequent motion and <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadence</a> helpers refine translation using
stationary stars and retain their registration scatter in the motion error.
This checks alignment in the actual image evidence rather than treating the
header as a guarantee of a perfect <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a>.

Arrays use [row, column] = [y, x], while source positions use (x, y).
[Astropy's WCS interface](https://docs.astropy.org/en/stable/wcs/wcsapi.html)
describes that distinction; [xRep](https://github.com/NVIDIA/cuPhoton/blob/main/docs/components/xrep.md)
documents mapping and plane propagation.

</details>

In [ ]:
from cuphoton.xrep import (
    BBox, ReprojectionSpec, make_wcs_pair_transform,
    reproject_masked_array,
)
# Approximate (x, y) search location, not the injected displacement.
old_guess = np.asarray(manifest["target_search"]["old_xy"], float)
# Current-position reference: steady light at the transit-era position.
trail_reference = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["reference_current"]))
trail_science = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["trail"]))
height, width = trail_reference["science"].shape
# Map destination/reference pixels to source/science coordinates.
mapping = make_wcs_pair_transform(
    WCS(trail_reference["header"]), WCS(trail_science["header"]))
# Reproject SCI [nJy], VAR [nJy²] and integer MASK together.
warped = reproject_masked_array(
    trail_science["science"], trail_science["variance"],
    trail_science["mask"],
    ReprojectionSpec(mapping=mapping,
                     output_bbox=BBox(0, 0, width, height),
                     mapping_grid_step=32), backend="cupy")
# Registered science still contains all its light; no subtraction yet.
aligned = dict(science=warped.image, variance=warped.variance,
               mask=warped.mask, header=trail_reference["header"].copy())

`aligned` now has the <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> shape and coordinate header. Its pixel values
still describe a <a class="model-sky-glossary-term" href="#model-sky-glossary-science-image">science exposure</a>. Persistent stars that fail to overlap
will leave structured <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a>, so the later motion helper checks their
alignment independently.

<a id="model-sky-match-the-blur-before-subtracting"></a>

### Match the blur before subtracting

Alignment fixes positions, but a stationary star can still have a different
shape in each image. The **<a class="model-sky-glossary-term" href="#model-sky-glossary-psf">point-spread function</a>**, or **PSF**, describes how
light from an <a class="model-sky-glossary-term" href="#model-sky-glossary-blend">unresolved</a> source spreads across pixels. Atmospheric turbulence
changes this blur, called **<a class="model-sky-glossary-term" href="#model-sky-glossary-seeing">seeing</a>**. Subtract unmatched stars and rings or
paired bright/dark edges can overwhelm the small change we want.

xPois fits a **<a class="model-sky-glossary-term" href="#model-sky-glossary-convolution">convolution</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-kernel">kernel</a>**, a small array of weights that blurs and
scales the <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> to match the <a class="model-sky-glossary-term" href="#model-sky-glossary-science-image">registered science</a> image, together with a
constant <a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a> offset. It learns from usable field pixels. We exclude
the target's search region so a changing target does not teach the fit to
remove its own signal. The first fit is provisional: it makes the satellite
trail easier to find before we exclude it and fit again.


<details class="model-sky-science"><summary>Science Explanation</summary>

Let R be the reference, S the registered science image, K the fitted kernel,
and B the fitted background offset. The matched reference M and difference D
are

<p class="model-sky-equation">M = K ⊗ R + B; &nbsp; D = S − M.</p>

Here ⊗ means convolution: each output pixel is a weighted <a class="model-sky-glossary-term" href="#model-sky-glossary-sum">sum</a> of nearby
reference pixels. R, S, M, B and D are in <a class="model-sky-glossary-term" href="#model-sky-glossary-njy">nJy</a>; K's weights are dimensionless.
Positive D means extra light in science; negative D means less light than in
the matched reference. The sign convention stays the same throughout.

The 21 × 21 kernel uses <a class="model-sky-glossary-term" href="#model-sky-glossary-gaussian">Gaussian-polynomial</a> basis functions at three widths.
Weighted <a class="model-sky-glossary-term" href="#model-sky-glossary-least-squares">least squares</a> selects their coefficients to reduce squared <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a>
on fit pixels, weighted here by science <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a>. `background_degree=0`
selects one constant offset. This call does not impose `flux_conserve=True`;
its fitted kernel can absorb <a class="model-sky-glossary-term" href="#model-sky-glossary-throughput">throughput</a> differences. It is not a deconvolution
that sharpens an arbitrarily blurred observation.

The analytic field already has a Gaussian PSF of width <a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">σ</a> = 1.5 pixels.
Preparation applies further seeing blur to both background and added stars
in each <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a>. In the continuous Gaussian approximation, blur variances
add: σ²<sub>effective</sub> = σ²<sub>base</sub> + σ²<sub>seeing</sub>.
The discrete shifted images need not be exact Gaussians. Measuring a sampled
PSF from image pixels keeps the fit tied to those images; evaluating it more
finely helps fractional shifts but creates no new telescope resolution. For method context,
see [Alard & Lupton (1998)](https://arxiv.org/abs/astro-ph/9712287).

</details>

In [ ]:
from cuphoton.xpois import GaussianBasisComponent, solve_constant_kernel
# Both frames must supply valid pixels; True means usable here.
usable = (pipeline.good_pixels(trail_reference)
          & pipeline.good_pixels(aligned))
# Withhold target light from learning blur and background.
fit_mask = usable & ~pipeline.exclusion_disks(usable.shape, [old_guess])
# Gaussian widths in pixels; polynomial degrees permit shape changes.
basis = [GaussianBasisComponent(sigma=0.7, degree=2),
         GaussianBasisComponent(sigma=1.6, degree=2),
         GaussianBasisComponent(sigma=3.0, degree=0)]
# A provisional residual exposes contamination for trail detection.
preliminary = solve_constant_kernel(
    trail_reference["science"], aligned["science"], basis,
    variance=aligned["variance"], fit_mask=fit_mask,
    kernel_shape=(21, 21), background_degree=0, backend="cupy")

`preliminary.matched` predicts the <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> under science-image blur and
<a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a>. `preliminary.residual` is science minus that prediction. The long
<a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> now supplies the evidence for proposing the trail <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a>.

<a id="model-sky-find-the-trail-exclude-it-refit"></a>

### Find the trail, exclude it, refit

The preliminary difference exposes a long coherent feature. A CPU
helper searches for aligned bright <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> pixels and estimates the trail's
width. This line detector is example code; xPois provides <a class="model-sky-glossary-term" href="#model-sky-glossary-kernel">kernel</a> fitting and
subtraction, not a satellite-identification service.

The next cell takes the proposed trail <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> back into xPois. Pixels marked
`True` in `trail_mask` are excluded; pixels marked `True` in `fit_mask` are
included. The refit estimates blur and <a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a> from the remaining field.
Its `matched` image predicts the steady sky under science-image conditions;
its `difference` is what remains after subtracting that prediction.


<details class="model-sky-science"><summary>Science Explanation</summary>

The helper uses a noise-scaled residual to propose a line, then measures its
profile perpendicular to that line. The initial <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-sum">sum</a> is a practical
detection approximation. The measurement helper subsequently propagates
<a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> variance through the squared fitted kernel to estimate <a class="model-sky-glossary-term" href="#model-sky-glossary-marginal">marginal</a>
residual variance. Neither treatment supplies full <a class="model-sky-glossary-term" href="#model-sky-glossary-interpolation">interpolation</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-covariance">covariance</a>
or all kernel-estimation <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a>.

For independent input pixel errors, the fixed-kernel approximation is

<p class="model-sky-equation">Var(D<sub>i</sub>) ≈ Var(S<sub>i</sub>) + <a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">Σ</a><sub>j</sub> K<sub>j</sub>² Var(R<sub>i−j</sub>).</p>

The sum runs over kernel pixels j; i labels an output pixel. The reference has
uncertainty even when it looks smooth. Additional model and resampling errors
need separate checks. Morphology alone does not prove an object's <a class="model-sky-glossary-term" href="#model-sky-glossary-orbit">orbit</a> or
identity. The Model Sky quality mask uses zero for usable pixels; the
Boolean trail and fit masks have the opposite inclusion meanings described above.

</details>

In [ ]:
# True means EXCLUDED here, opposite to the fit-mask convention.
trail_mask, trail_profile = pipeline.detect_trail(
    preliminary.residual,
    aligned["variance"] + trail_reference["variance"], usable)
# Refit the same arrays with trail pixels excluded.
kernel_fit = solve_constant_kernel(
    trail_reference["science"], aligned["science"], basis,
    variance=aligned["variance"], fit_mask=fit_mask & ~trail_mask,
    kernel_shape=(21, 21), background_degree=0, backend="cupy")
# Distinct products: registered science, matched reference, difference.
trail = dict(science=aligned["science"],
             reference=trail_reference["science"],
             difference=kernel_fit.residual, matched=kernel_fit.matched,
             trail_mask=trail_mask, trail_profile=trail_profile,
             fit_mask=kernel_fit.fit_mask, kernel=kernel_fit.kernel,
             registered=aligned)
# Profile coordinates run across the trail rather than along it.
profile = (dict(x=trail_profile["coordinate"],
                difference=trail_profile["profile"],
                x_label="Perpendicular distance from trail [pixel]",
                y_label="Median residual [nJy]")
           if trail_profile["detected"] else None)
show_dark(views.trail_view(
    trail["science"], trail["matched"], trail["difference"], trail_mask,
    profile=profile, pixel_scale=scale,
    wcs=WCS(trail_reference["header"]),
    context="Amber marks excluded pixels in the controlled scene."))

<a id="model-sky-exercise-how-wide-should-the-mask-be"></a>

### Exercise · How wide should the mask be?

In the figure, amber marks rejected pixels. Toggle the <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> and blink the
images to see the tradeoff: unmasked wings can bias a fit, while an excessively
wide mask throws away useful stars and <a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a>. The perpendicular profile
helps distinguish the bright core from fainter trail wings.

Predict what will happen at `width_scale = 0.7` and `1.5`, then compare the
number of fit pixels and <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-rms">RMS</a>, the square root of the average squared
residual. Use the **same evaluation region** for both trials. Otherwise a lower
RMS could simply mean you stopped counting the difficult pixels.


<details class="model-sky-science"><summary>Science Explanation</summary>

For N evaluation pixels with differences D<sub>i</sub> in <a class="model-sky-glossary-term" href="#model-sky-glossary-njy">nJy</a>,

<p class="model-sky-equation">RMS = √[(1/N) <a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">Σ</a><sub>i</sub> D<sub>i</sub>²].</p>

This describes residual size, not the probability that a model is correct.
A structured leftover can matter even when average RMS is small. The exercise
excludes the widest trial mask from evaluation for every setting, so the
statistic compares the same sky pixels.

</details>

In [ ]:
width_scale = 1.0  # Try 0.7 and 1.5 after the first complete run.
trial_mask, _ = pipeline.detect_trail(
    preliminary.residual,
    aligned["variance"] + trail_reference["variance"], usable,
    width_scale=width_scale)
trial = solve_constant_kernel(
    trail_reference["science"], aligned["science"], basis,
    variance=aligned["variance"], fit_mask=fit_mask & ~trial_mask,
    kernel_shape=(21, 21), background_degree=0, backend="cupy")
wide_mask, _ = pipeline.detect_trail(
    preliminary.residual,
    aligned["variance"] + trail_reference["variance"], usable,
    width_scale=1.5)
# Compare all mask choices on this same fixed region.
evaluation = fit_mask & ~wide_mask
print("Fit pixels:", np.count_nonzero(fit_mask & ~trial_mask))
print("Common-region residual RMS [nJy]:",
      np.sqrt(np.mean(trial.residual[evaluation] ** 2)))

A useful <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> removes contaminated wings without discarding unnecessary
information. Compare the printed common-region <a class="model-sky-glossary-term" href="#model-sky-glossary-rms">RMS</a> with the image structure;
neither fit-pixel count nor RMS alone chooses the scientific answer.

<details><summary>Image processing from a terminal</summary>

Run these recipes after Chapter 2 saves the motion artifacts:

```bash
cuphoton xrep reproject-stack --inputs data/motion/science.fits \
  --hdu 1 --target-wcs data/refs/reference_old.fits --target-hdu 1 \
  --backend cupy --fits-reader astropy --repeats 1 --warmup 0 \
  --write-fits --name motion-grid --output-dir results/xrep
cuphoton xpois fit-kernel \
  --reference results/motion_reference.fits --reference-hdu 1 \
  --target results/motion_registered.fits --target-hdu 1 \
  --variance results/motion_registered.fits --variance-hdu 2 \
  --fit-mask results/motion_fit_mask.npy \
  --basis-sigmas 0.7,1.6,3 --basis-degrees 2,2,0 \
  --kernel-height 21 --kernel-width 21 --background-degree 0 \
  --backend cupy --fits-reader astropy --no-review \
  --name motion-match --output-dir results/xpois
```

These use the bundle's <a class="model-sky-glossary-term" href="#model-sky-glossary-science-image">science image</a> and saved fit inputs. See
[xRep](https://github.com/NVIDIA/cuPhoton/blob/main/docs/components/xrep.md)
and [xPois](https://github.com/NVIDIA/cuPhoton/blob/main/docs/components/xpois.md)
for plane selection and artifacts.

</details>

<a id="model-sky-exercise-what-if-the-reference-comes-from-the-wrong-epoch"></a>

### Exercise · What if the reference comes from the wrong epoch?

A <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> answers a specific question. For trail rejection and a <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>
series, we want the star at its current position. For motion, we deliberately
compare its earlier and later positions. An **<a class="model-sky-glossary-term" href="#model-sky-glossary-epoch">epoch</a>** is the time an
observation represents.

The next cell substitutes the old-position reference in the trail example.
Inspect the target cutout: a bright and a dark patch now survive subtraction
even though neither is a new object. This is a **<a class="model-sky-glossary-term" href="#model-sky-glossary-dipole">dipole</a>**, the two-lobed
footprint of light disappearing at one position and appearing at another.


<details class="model-sky-science"><summary>Science Explanation</summary>

A difference records change relative to the chosen reference. It cannot say
which reference was appropriate for the question. A motion reference encodes
the old position; a current reference removes steady light at the later
position so a transit becomes a negative <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> there. A <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a> error
can also make dipoles around stationary objects. Neighboring field stars
distinguish a global alignment problem from one source's <a class="model-sky-glossary-term" href="#model-sky-glossary-displacement">displacement</a>.

</details>

In [ ]:
earlier_reference = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["reference_old"]))
# Deliberately change the comparison epoch, not the image grid.
wrong_reference = pipeline.subtract(
    earlier_reference, aligned, exclude=[old_guess])
x, y = np.rint(old_guess).astype(int)
# Array slices use [y, x]; source positions use (x, y).
region = (slice(y - 35, y + 36), slice(x - 35, x + 36))
show_dark(views.image_triptych(
    wrong_reference["science"][region],
    wrong_reference["matched"][region],
    wrong_reference["difference"][region],
    title="Earlier reference · target residual", pixel_scale=scale,
    wcs=WCS(earlier_reference["header"]).slice(region)))

<a id="model-sky-2-the-star-moves-the-field-stays-put"></a>

## 2 · The star moves; the field stays put

The old and new target positions now become the signal we want. The motion
helper registers the later <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a> using neighboring stars, matches the old
<a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a>'s blur, and fits the two-lobed <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> with xFit. A **<a class="model-sky-glossary-term" href="#model-sky-glossary-lobe">lobe</a>** is one
bright or dark patch; a **<a class="model-sky-glossary-term" href="#model-sky-glossary-dipole">dipole</a>** is the pair. The negative lobe marks the old
position, and the positive lobe marks the new one.

A stationary neighbor should largely cancel. Dipoles around many neighbors
would instead implicate <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a>. The helper measures independent compact
field stars to refine alignment and estimates a registration error floor from
their scatter. It also checks that the fitted old lobe belongs to the searched
star, rather than attaching an unrelated residual to it.

The output is a <a class="model-sky-glossary-term" href="#model-sky-glossary-displacement">displacement</a> in pixels and an <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a>. Converting it to
**<a class="model-sky-glossary-term" href="#model-sky-glossary-proper-motion">proper motion</a>**, angular travel per year across the sky, requires a pixel
scale and time interval. The ten-year interval is a declared part of this simulation, not the
elapsed duration of notebook execution or a measured observing campaign.


<details class="model-sky-science"><summary>Science Explanation</summary>

For fitted positions (x₊, y₊) and (x₋, y₋), the displacement length d is

<p class="model-sky-equation">d = √[(x₊ − x₋)² + (y₊ − y₋)²]; &nbsp; μ = 1000 s d / Δt.</p>

Coordinates and d are in pixels, s is arcsec/pixel, Δt is years, and μ is
milliarcseconds/year (mas/year). One <a class="model-sky-glossary-term" href="#model-sky-glossary-arcsecond">arcsecond</a> is 1/3600 degree; a
milliarcsecond is one thousandth of that. The scalar scale is the declared 0.2 arcsec/pixel
scale, used locally across this small modeled field.

The sampled <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> comes from the matched reference. xFit estimates both centers
and a shared amplitude; its <a class="model-sky-glossary-term" href="#model-sky-glossary-covariance">covariance</a> describes how their fitted errors vary
together. The helper adds a field-registration floor in <a class="model-sky-glossary-term" href="#model-sky-glossary-quadrature">quadrature</a>:
<a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">σ</a><sub>d</sub> = √(σ<sub>fit</sub>² + 2 σ<sub>reg</sub>²). Both terms are in
pixels: σ<sub>fit</sub> is the fit error and σ<sub>reg</sub> is stationary-star
alignment scatter. These are <a class="model-sky-glossary-term" href="#model-sky-glossary-conditional">conditional</a>
uncertainties, not a complete survey <a class="model-sky-glossary-term" href="#model-sky-glossary-astrometry">astrometric</a> error budget.

Proper motion reflects relative travel across the sky. **<a class="model-sky-glossary-term" href="#model-sky-glossary-parallax">Parallax</a>** is the
apparent annual shift caused by our changing viewpoint as Earth orbits the
Sun. Two simulated <a class="model-sky-glossary-term" href="#model-sky-glossary-epoch">epochs</a> here do not determine parallax, acceleration,
distance or a three-dimensional space velocity. [ESA's Gaia overview](https://www.esa.int/articles/about-us/directorates/science/gaia/how-does-gaia-study-the-milky-way)
explains the distinction. The [GJ 1214 discovery](https://arxiv.org/abs/0912.3229)
shows how motion and archival images can help rule out <a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-blend">blends</a>.

</details>

In [ ]:
reference_old = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["reference_old"]))
motion_science = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["motion"]))
# Field stars refine alignment; xFit measures a PSF-shaped dipole.
motion = pipeline.measure_motion(
    reference_old, motion_science, old_guess,
    search_radius=manifest["target_search"]["radius_pixels"])
# Simulated elapsed time between the modeled observing epochs.
baseline = float(assumptions["baseline_years"])
# Pixels × arcsec/pixel × 1000 mas/arcsec ÷ elapsed years.
motion_masyr = motion["separation_pixels"] * scale * 1000 / baseline
motion_error = motion["separation_error_pixels"] * scale * 1000 / baseline
print(f"Motion: {motion_masyr:.2f} ± {motion_error:.2f} mas/year")
print("Registration floor [pixel]:", motion["registration_floor_pixels"])

The printed motion includes dipole-fit error and the <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a> floor.
The next view lets you check its spatial interpretation rather than accepting
a successful numerical fit as sufficient evidence.

In [ ]:
show_dark(views.motion_view(
    motion["matched"], motion["science"], motion["difference"],
    old_xy=motion["old_xy"], new_xy=motion["new_xy"],
    baseline_years=baseline, pixel_scale=scale,
    wcs=WCS(reference_old["header"]),
    epoch_labels=("Injected earlier position", "Injected later position"),
    context="Independent stationary stars set registration."))
# Save exact inputs for the terminal recipes below.
workshop.save_motion(OUTPUT, motion, reference_old)

<a id="model-sky-exercise-is-the-dipole-measurement-stable"></a>

### Exercise · Is the dipole measurement stable?

Blink the motion figure: only the <a class="model-sky-glossary-term" href="#model-sky-glossary-ground-truth">injected</a> star should visibly move after
alignment. Follow the <a class="model-sky-glossary-term" href="#model-sky-glossary-lobe">negative lobe</a> to the old location and the positive lobe
to the new one. The displayed motion includes <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a>; a
direct cutout fit alone does not.

The next cell exposes the actual xFit call. A **<a class="model-sky-glossary-term" href="#model-sky-glossary-stamp">stamp</a>** is a small image
cutout around a source. Its measured <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> is sampled at quarter-pixel spacing
to evaluate fractional shifts accurately. Fit the unchanged stamp first, then
optionally trim two or four pixels from its edge. Preserve both lobes and
compare the result with the original uncertainty.


<details class="model-sky-science"><summary>Science Explanation</summary>

The model is a positive shifted PSF minus a negative shifted copy:

<p class="model-sky-equation">D(x, y) ≈ A [P(x − x₊, y − y₊) − P(x − x₋, y − y₋)].</p>

P is the dimensionless sampled PSF, A its fitted amplitude in <a class="model-sky-glossary-term" href="#model-sky-glossary-njy">nJy</a>, and the four
center coordinates are in pixels relative to the stamp center. The fit
weights <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a> by <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a> and ignores excluded pixels. A free <a class="model-sky-glossary-term" href="#model-sky-glossary-gaussian">Gaussian</a>
width can trade against lobe separation when the lobes overlap. Even with
an analytic Gaussian starting profile, <a class="model-sky-glossary-term" href="#model-sky-glossary-seeing">seeing</a>, pixel <a class="model-sky-glossary-term" href="#model-sky-glossary-sampling">sampling</a> and reprojection
alter the observed shape. A fixed sampled PSF measured from the matched
<a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> avoids introducing a free width into this <a class="model-sky-glossary-term" href="#model-sky-glossary-dipole">dipole</a> fit; it does not
make the measured PSF noise-free.

If C is the parameter <a class="model-sky-glossary-term" href="#model-sky-glossary-covariance">covariance</a> and g the <a class="model-sky-glossary-term" href="#model-sky-glossary-gradient">gradient</a> of separation with respect
to those parameters, the local fit error is σ²<sub>d,fit</sub> = gᵀCg. A
gradient describes how much a result changes when each parameter changes
slightly. This includes correlations between center coordinates. Convergence
means the solver stopped successfully, not that the model is scientifically
adequate. Check residuals, association and registration evidence too. See
[xFit's model documentation](https://github.com/NVIDIA/cuPhoton/blob/main/docs/components/xfit.md).

</details>

In [ ]:
from cuphoton.xfit import StampDipoleModel, fit_dipoles
# One residual stamp: (batch=1, height, width).
dipole_input = motion["fit_input"]
dipole_mask = dipole_input["mask"].copy()
trim_pixels = 0  # Try 2 or 4; preserve both source lobes.
if trim_pixels:
    dipole_mask[:, :trim_pixels, :] = False
    dipole_mask[:, -trim_pixels:, :] = False
    dipole_mask[:, :, :trim_pixels] = False
    dipole_mask[:, :, -trim_pixels:] = False
# PSF measured from the matched reference, not injection truth.
fit = fit_dipoles(
    dipole_input["images"],
    model=StampDipoleModel(dipole_input["stamp_basis"][0],
                          scale=float(dipole_input["stamp_scale"]),
                          image_shape=dipole_input["images"].shape[-2:],
                          backend="cupy"),
    variance=dipole_input["variance"], mask=dipole_mask, backend="cupy")
# Parameters: positive (x,y), negative (x,y), shared amplitude.
displacement = fit.parameters[0, :2] - fit.parameters[0, 2:4]
separation = np.linalg.norm(displacement)
# Propagate full covariance into scalar separation.
gradient = np.r_[displacement, -displacement, 0.] / separation
separation_error = np.sqrt(gradient @ fit.covariance[0] @ gradient)
print("Converged:", bool(fit.converged[0]))
print("Separation and formal fit error [pixel]:",
      float(separation), float(separation_error))
# The direct fit error excludes the separate registration floor above.

<details><summary>Motion measurement from a terminal</summary>

The notebook saves <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a>, <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variance</a> and valid-pixel <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">mask</a> as a numeric
NPZ input for the standard xFit interface:

```bash
cuphoton xfit fit-dipoles --input results/motion_candidates.npz \
  --model stamp --stamp-scale 0.25 --backend cupy \
  --fits-reader astropy \
  --output-dir results/fitted-dipole
```

[xFit documentation](https://github.com/NVIDIA/cuPhoton/blob/main/docs/components/xfit.md)
explains derived quantities. A <a class="model-sky-glossary-term" href="#model-sky-glossary-dipole">dipole</a> fit alone does not identify a
star, asteroid or <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a> artifact; time and field evidence
supply that interpretation.

</details>

<a id="model-sky-3-a-smaller-shadow-on-the-same-star"></a>

## 3 · A smaller shadow, on the same star

We know where the star is now. Next we measure its light repeatedly at that
position, looking for a short decrease. **<a class="model-sky-glossary-term" href="#model-sky-glossary-forced-photometry">Forced photometry</a>** fits the amount
of light at an already measured position, even when a faint <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> would
not trigger a new detection. Fitting a fresh position in every noisy <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a>
would give <a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a> another way to change the measured brightness.

The fictional system is inspired by GJ 1214, a small cool star with a planet
found using ground-based observations. We place a similar star at 146.42
<a class="model-sky-glossary-term" href="#model-sky-glossary-parsec">parsecs</a> and choose 120 exposures, each 30 seconds long, one minute apart.
A parsec is a distance unit, about 3.26 <a class="model-sky-glossary-term" href="#model-sky-glossary-light-year">light-years</a>. The roughly 52-minute
<a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a> and dense <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadence</a> are designed for the exercise. They are simulated
observations of our fictional system, not observations of GJ 1214.

The next cell refines the recovered position in an independent current-position
<a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a>. It registers the first exposure, estimates its <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> from neighboring
stars, removes a local <a class="model-sky-glossary-term" href="#model-sky-glossary-background">background</a>, and measures target <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a>. The output is a
brightness in <a class="model-sky-glossary-term" href="#model-sky-glossary-njy">nJy</a> and an error, not yet a planet radius.


<details class="model-sky-science"><summary>Science Explanation</summary>

For background-subtracted <a class="model-sky-glossary-term" href="#model-sky-glossary-stamp">stamp</a> pixels I<sub>i</sub>, a fixed <a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalized</a> PSF
P<sub>i</sub>, and <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variances</a> V<sub>i</sub>, forced PSF photometry is a weighted
amplitude fit. In the independent-pixel approximation,

<p class="model-sky-equation">F = [<a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">Σ</a><sub>i</sub> P<sub>i</sub>I<sub>i</sub>/V<sub>i</sub>] / [Σ<sub>i</sub> P<sub>i</sub>²/V<sub>i</sub>]; &nbsp; σ²<sub>F</sub> = 1 / [Σ<sub>i</sub> P<sub>i</sub>²/V<sub>i</sub>].</p>

F and σ<sub>F</sub> are in nJy when P sums to one; sums include only usable
pixels. The <a class="model-sky-glossary-term" href="#model-sky-glossary-conditional">conditional error</a> treats position, PSF and local sky estimate as
fixed. The local sky is estimated from an outer <a class="model-sky-glossary-term" href="#model-sky-glossary-aperture">annulus</a>, a ring around the
source cutout. Imperfect PSFs, neighbors or <a class="model-sky-glossary-term" href="#model-sky-glossary-interpolation">interpolation</a> can add errors beyond
this formula.

**From a model star to detected <a class="model-sky-glossary-term" href="#model-sky-glossary-electron">electrons</a>.** We specify a 3100 K <a class="model-sky-glossary-term" href="#model-sky-glossary-blackbody">blackbody</a>,
radius R = 0.2162 <a class="model-sky-glossary-term" href="#model-sky-glossary-stellar-radius">solar radii</a> and distance d = 146.42 parsecs. A blackbody is
an ideal thermal <a class="model-sky-glossary-term" href="#model-sky-glossary-spectrum">spectrum</a>; its Planck radiance B<sub>λ</sub> depends on
<a class="model-sky-glossary-term" href="#model-sky-glossary-wavelength">wavelength</a> λ and temperature T. With h the Planck constant, c the speed of
light and k<sub>B</sub> the Boltzmann constant, preparation computes

<p class="model-sky-equation">B<sub>λ</sub>(T) = (2hc²/λ⁵) / [exp(hc/(λk<sub>B</sub>T)) − 1]; &nbsp; F<sub>λ</sub> = π B<sub>λ</sub>(R/d)².</p>

F<sub>λ</sub> is the spectral flux at Earth in W m⁻² m⁻¹ when SI units are
used. The squared radius-to-distance ratio expresses geometric dilution.
<a class="model-sky-glossary-term" href="#model-sky-glossary-throughput">Throughput</a> η(λ) is the fraction of <a class="model-sky-glossary-term" href="#model-sky-glossary-photon">photons</a> detected: here 0.4 in the declared
560–700 nm band. A photon has energy hc/λ. The 4 m diameter telescope has
collecting area A = π(2 m)² ≈ 12.566 m²; this area and exposure duration t
convert the spectrum into an expected electron count:

<p class="model-sky-equation">N<sub>e</sub> = At ∫ F<sub>λ</sub> η(λ) λ/(hc) dλ.</p>

For comparison with <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">calibrated</a> image flux, the photon-weighted spectral flux
density is f̄<sub>ν</sub> = ∫F<sub>λ</sub> λη(λ)dλ / [c ∫η(λ)dλ/λ].
The declared model gives **348775 nJy**, **<a class="model-sky-glossary-term" href="#model-sky-glossary-magnitude">AB magnitude</a> 17.54**, and about
**177119 target electrons in 30 seconds** before exposure-specific modulation.
The AB conversion is m<sub>AB</sub> = −2.5 log₁₀[f̄<sub>ν</sub>/(3631 Jy)];
lower magnitudes mean brighter sources. This uses the <a class="model-sky-glossary-term" href="#model-sky-glossary-bandpass">modeled band</a>, not a
measured r-filter response.

A cool M dwarf has molecular absorption absent from this smooth continuum.
<a class="model-sky-glossary-term" href="#model-sky-glossary-extinction">Extinction</a> by intervening material is also omitted. The brightness and
precision are therefore predictions of this explicit teaching model, not of
a real telescope observing GJ 1214. The <a class="model-sky-glossary-term" href="#model-sky-glossary-limb-darkening">limb-darkening</a> law used below is a
separately declared illustrative surface-brightness pattern; a uniform
blackbody does not derive those coefficients. See
[Astropy's Planck-model documentation](https://docs.astropy.org/en/stable/modeling/physical_models.html#blackbody)
and the [recorded system reference](https://arxiv.org/html/2402.05991v2).

</details>

In [ ]:
from cuphoton.xfit import forced_psf_photometry
from scipy.ndimage import shift

# Transit uses the current-position reference; motion uses the older one.
current_reference = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["files"]["reference_current"]))
# Start at the recovered position and refine once in the reference.
target_xy = motion["new_xy"]
target_fit, _, target_ok = pipeline.fit_positions(
    current_reference, np.array([target_xy]))
assert target_ok[0], "Target position must be measured before photometry"
target_xy = target_fit[0]
# Other stars define alignment and PSF independently of target variability.
anchors = pipeline.select_stars(current_reference, exclude=[target_xy])
first_exposure = pipeline.read_exposure(
    pipeline.bundle_path(root, manifest["epochs"][0]["path"]))
first_aligned = pipeline.register(
    first_exposure, current_reference, anchors=anchors)
psf = pipeline.empirical_psf(first_aligned, anchors)
# Background-subtracted science stamp [nJy]; variance stays in nJy².
patch = pipeline.stamp(first_aligned["science"], target_xy)
patch -= pipeline.local_sky(patch)
patch_variance = pipeline.stamp(first_aligned["variance"], target_xy)
patch_mask = pipeline.stamp(
    pipeline.good_pixels(first_aligned), target_xy)
# Subpixel (x,y) offset; scipy shifts use array order (y,x).
offset = target_xy - np.rint(target_xy)
shifted_psf = shift(psf, offset[::-1], order=3)
# Add batch dimension; a fixed measured PSF leaves flux as the unknown.
one_flux = forced_psf_photometry(
    cp.asarray(patch[None]), cp.asarray(patch_variance[None]),
    cp.asarray(offset[None]), 1, mask=cp.asarray(patch_mask[None]),
    psf=cp.asarray(shifted_psf[None]))
print("First-exposure flux and conditional error [nJy]:",
      float(one_flux.flux[0]), float(one_flux.flux_error[0]))

The first-exposure result measures brightness under a fixed position and <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a>
model. Its <a class="model-sky-glossary-term" href="#model-sky-glossary-conditional">conditional error</a> describes fluctuations within that model. The
time series and <a class="model-sky-glossary-term" href="#model-sky-glossary-comparison-star">comparison stars</a> will help separate a brief target dimming
from changes affecting the whole frame.

<a id="model-sky-build-a-light-curve-without-letting-the-reference-set-the-answer"></a>

### Build a light curve without letting the reference set the answer

One <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a> cannot reveal a <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>. The next helper repeats <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a>, trail
<a class="model-sky-glossary-term" href="#model-sky-glossary-mask">masking</a>, <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF matching</a> and <a class="model-sky-glossary-term" href="#model-sky-glossary-photometry">photometry</a> across the sequence. It batches the target,
a <a class="model-sky-glossary-term" href="#model-sky-glossary-check-star">check star</a> and <a class="model-sky-glossary-term" href="#model-sky-glossary-comparison-star">comparison stars</a> within each <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a>. It returns time-indexed
fluxes, errors, validity flags and image <a class="model-sky-glossary-term" href="#model-sky-glossary-stamp">stamps</a>, then fits the <a class="model-sky-glossary-term" href="#model-sky-glossary-light-curve">light curve</a>.

The current <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> matters: the star is already at the position we monitor.
At each <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadence</a>, the pipeline measures **reference flux + <a class="model-sky-glossary-term" href="#model-sky-glossary-difference">difference flux</a>**
with the same PSF, weights and pixel mask. It divides the target's relative
brightness by a weighted comparison-star ensemble. Stars that should stay
steady reveal common transmission changes. A separate check star stays out
of that ensemble to test the remaining scatter.

A fitted flux can still be unreliable if a trail removes the star's core.
The helper requires at least 90% of the PSF's light to survive the mask.
A rejected cadence is unavailable information, not zero brightness.


<details class="model-sky-science"><summary>Science Explanation</summary>

Write the linear flux estimator as L. With the same weights and masks,

<p class="model-sky-equation">F<sub>total</sub> = L(M) + L(S − M) = L(S).</p>

The matched-reference contribution cancels algebraically. Adding the two
formal <a class="model-sky-glossary-term" href="#model-sky-glossary-variance">variances</a> as if these terms were independent would double-count shared
reference <a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a>. The implementation therefore uses science-image variance
for total-flux error. This cancellation does not erase errors in estimated
positions, PSFs, sky or <a class="model-sky-glossary-term" href="#model-sky-glossary-kernel">kernel</a> decisions. Here the latent analytic sky is
noiseless, but estimated quantities still depend on finite-noise reference
images that are reused across the sequence. The fixed-pixel algebra does not
turn those estimated quantities into exact knowledge.

For comparison-star relative fluxes f<sub>j</sub> and errors <a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">σ</a><sub>j</sub>,
the ensemble E = Σ(w<sub>j</sub>f<sub>j</sub>)/Σw<sub>j</sub>, with
w<sub>j</sub> = 1/σ<sub>j</sub>². The <a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalized</a> target is f<sub>target</sub>/E.
The helper propagates target and ensemble errors, then increases the error
scale if robust check-star scatter exceeds the formal expectation. It never
shrinks the scale below one. This <a class="model-sky-glossary-term" href="#model-sky-glossary-empirical">empirical</a> <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">calibration</a> does not supply a
full model of time-correlated <a class="model-sky-glossary-term" href="#model-sky-glossary-systematics">systematics</a>.

</details>

In [ ]:
def progress(done, total):
    print(f"Measured {done}/{total} cadences",
          end="\r" if done < total else "\n")

# Per-cadence target/check/comparison fluxes, errors and quality flags.
transit = pipeline.measure_transit(
    root, manifest, target_xy, progress=progress)
workshop.save_transit(OUTPUT, transit)
print(f"{transit['valid'].sum()} usable cadences · "
      f"{transit['elapsed_seconds']:.1f} s · "
      f"error scale {transit['error_scale']:.2f}")

<a id="model-sky-read-the-dip-and-the-images-together"></a>

### Read the dip and the images together

The progress output reports usable <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadences</a> and the <a class="model-sky-glossary-term" href="#model-sky-glossary-empirical">empirical</a> error scale.
The reference execution should retain 119 of 120 cadences: the trail-crossed
<a class="model-sky-glossary-term" href="#model-sky-glossary-exposure">exposure</a> is rejected. A different count deserves inspection before interpreting
a fit. Time spent here describes this run and cache state, not a benchmark.

In the next figure, tap a light-curve point or move the cadence slider. The
native image <a class="model-sky-glossary-term" href="#model-sky-glossary-stamp">stamp</a> and difference stamp update together. During the <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>,
missing light appears negative at the recovered star position. The <a class="model-sky-glossary-term" href="#model-sky-glossary-check-star">check-star</a>
curve should not share the target's dip. Inspect fit <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a>, measured <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a>
minus the model, for leftover trends or clusters.

The stacked difference subtracts a mean baseline residual from a mean
in-transit residual. It helps locate missing light and suppress persistent
<a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> mismatch. The fitted model selects those intervals, so this panel is a
visualization of the fit rather than an independent detection statistic.


<details class="model-sky-science"><summary>Science Explanation</summary>

**A finite-exposure transit model.** A planet blocks part of the stellar disk.
Stars generally look dimmer near the visible edge, an effect called limb
darkening. The adopted quadratic law is

<p class="model-sky-equation">I(q)/I(1) = 1 − u₁(1 − q) − u₂(1 − q)².</p>

I is surface brightness, q is the <a class="model-sky-glossary-term" href="#model-sky-glossary-cosine">cosine</a> of the angle between the local
surface normal and our <a class="model-sky-glossary-term" href="#model-sky-glossary-sky-plane">line of sight</a> (one at disk center, zero at the edge),
and u₁, u₂ are dimensionless coefficients. Here u₁ = 0.6 and u₂ = 0.15
are declared illustrative values. They are neither atmosphere-table values
for a measured filter nor a consequence of the <a class="model-sky-glossary-term" href="#model-sky-glossary-blackbody">blackbody</a> continuum. The
intensity is <a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalized</a> to one at disk center; the adopted law gives 0.25 at
the edge. Changing it tests how inferred radius depends on assumed stellar
brightness structure.

The model averages blocked light over each 30-second exposure, smoothing rapid
ingress and egress, the beginning and end of the crossing. `batman` uses seven
sub-times per exposure in the fit; the generator uses eleven. The fit varies
<a class="model-sky-glossary-term" href="#model-sky-glossary-radius-ratio">radius ratio</a> k = R<sub>p</sub>/R<sub>★</sub>, event midpoint t₀, and two
baseline light-curve terms: a normalization and a linear time slope. <a class="model-sky-glossary-term" href="#model-sky-glossary-orbit">Period</a>,
orbital geometry and <a class="model-sky-glossary-term" href="#model-sky-glossary-limb-darkening">limb darkening</a> stay fixed at declared assumptions.

For a uniform disk and a small fully overlapping planet, fractional depth is
approximately k². With limb darkening, peak measured dimming also depends on
which stellar surface the planet crosses. A physical planet radius requires
R<sub>p</sub> = k R<sub>★</sub>, hence an assumed or independently measured
<a class="model-sky-glossary-term" href="#model-sky-glossary-stellar-radius">stellar radius</a>. One event does not measure the orbital period. Reported
parameter errors condition on the fixed assumptions and adopted <a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a> model.

See [Mandel & Agol (2002)](https://arxiv.org/abs/astro-ph/0210099),
[Kreidberg's batman paper](https://arxiv.org/abs/1507.08285) and
[the batman tutorial](https://lkreidberg.github.io/batman/docs/html/tutorial.html).

</details>

In [ ]:
transit_fit = transit["fit"]
tx, ty = np.rint(transit["positions"][0]).astype(int)
half = transit["stamps"].shape[-1] // 2
# Preserve sky coordinates for the native-pixel target cutout.
transit_wcs = WCS(current_reference["header"]).slice(
    (slice(ty - half, ty + half + 1), slice(tx - half, tx + half + 1)))
show_dark(views.transit_view(
    transit["times_minutes"], transit["flux"], transit["flux_error"],
    transit_fit["model"], stamps=transit["stamps"],
    transit_model=transit_fit["transit_model"],
    difference_stamps=transit["difference_stamps"],
    valid=transit["valid"], check_flux=transit["check_flux"],
    reference_stamp=transit["reference_stamp"], pixel_scale=scale,
    wcs=transit_wcs,
    summary={
        "Radius ratio": f"{transit_fit['radius_ratio']:.4f}",
        "Conditional 1σ": f"{transit_fit['radius_ratio_error']:.4f}",
        "Mid-transit [min]": f"{transit_fit['midtime_min']:.2f}",
    }))

<a id="model-sky-exercise-which-comparison-stars-help"></a>

### Exercise · Which comparison stars help?

Reuse the measured <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a> table to change the <a class="model-sky-glossary-term" href="#model-sky-glossary-comparison-star">comparison ensemble</a>. Columns are
target, <a class="model-sky-glossary-term" href="#model-sky-glossary-check-star">check star</a>, then comparison stars; rows are <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadences</a>. Changing
`comparison_count` changes the common-brightness correction without rerunning
images. More stars help only if their measurements are useful.

Predict whether using one rather than five comparisons will change the fitted
<a class="model-sky-glossary-term" href="#model-sky-glossary-radius-ratio">radius ratio</a> or <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residual</a> scatter. Check whether the held-out star also becomes
steadier. A favorable fit to the target alone is weak evidence for a better
<a class="model-sky-glossary-term" href="#model-sky-glossary-normalization">normalization</a>, because the target contains the signal we want to preserve.


<details class="model-sky-science"><summary>Science Explanation</summary>

Inverse-variance weighting gives more influence to a precise comparison star.
It assumes quoted errors describe useful scatter. A variable, blended or
systematically biased comparison can make the correction worse despite a
small <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">formal error</a>. Dividing every star by its own median first puts them on
a common dimensionless scale. The check star tests the ensemble without taking
part in its weighted average.

The exercise retains the original quality flags and <a class="model-sky-glossary-term" href="#model-sky-glossary-empirical">empirical</a> error scale.
It changes one analysis choice rather than establishing a new independent
uncertainty <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">calibration</a>.

</details>

In [ ]:
comparison_count = 3  # Choose 1–5 from the measured ensemble.
# Columns: target, check star, comparisons. Rows: cadences.
raw, raw_error = transit["raw_flux"], transit["raw_error"]
# Normalize each star by its own median before averaging comparisons.
med = np.nanmedian(raw, axis=0)
relative, relative_error = raw / med, raw_error / med
# Skip target and held-out check columns in the correction ensemble.
selected = slice(2, 2 + comparison_count)
comp, comp_error = relative[:, selected], relative_error[:, selected]
good = np.isfinite(comp) & np.isfinite(comp_error) & (comp_error > 0)
# Inverse-variance weights: uncertain comparisons contribute less.
weights = np.where(good, 1 / comp_error**2, 0)
weight_sum = weights.sum(axis=1)
ensemble = np.sum(np.where(good, comp, 0) * weights, axis=1) / weight_sum
trial_flux = relative[:, 0] / ensemble
# Include target and ensemble noise; retain the original error scale.
trial_error = np.hypot(relative_error[:, 0] / ensemble,
                       trial_flux / np.sqrt(weight_sum) / ensemble)
trial_valid = transit["valid"] & (weight_sum > 0)
# Normalize without selecting epochs using hidden truth.
baseline_flux = np.nanpercentile(trial_flux[trial_valid], 70)
trial_flux /= baseline_flux
trial_error *= transit["error_scale"] / baseline_flux
trial_fit = pipeline.fit_transit(
    transit["times_minutes"], trial_flux, trial_error,
    trial_valid, assumptions)
print("Radius ratio:", trial_fit["radius_ratio"])
print("Residual scatter:",
      np.nanstd((trial_flux - trial_fit["model"])[trial_valid]))

Compare this trial's <a class="model-sky-glossary-term" href="#model-sky-glossary-radius-ratio">radius ratio</a> and scatter with the original result. If the
shift exceeds the quoted error, inspect <a class="model-sky-glossary-term" href="#model-sky-glossary-comparison-star">comparison-star</a> behavior and <a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a>
before attributing the change to the planet.

<a id="model-sky-exercise-what-do-we-know-about-the-planet"></a>

### Exercise · What do we know about the planet?

Change the assumed <a class="model-sky-glossary-term" href="#model-sky-glossary-limb-darkening">limb darkening</a> and refit the same measured <a class="model-sky-glossary-term" href="#model-sky-glossary-light-curve">light curve</a>.
Compare the radius-ratio shift with the original <a class="model-sky-glossary-term" href="#model-sky-glossary-conditional">conditional error</a>. A precise
answer under one assumed star model can move when that model changes.

The data directly constrain a dimming pattern and its timing. The planet-size
interpretation additionally depends on stellar brightness structure, crossing
geometry and <a class="model-sky-glossary-term" href="#model-sky-glossary-stellar-radius">stellar radius</a>. A small error printed by an optimizer does not
include all of those <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainties</a>.


<details class="model-sky-science"><summary>Science Explanation</summary>

If k and stellar radius R<sub>★</sub> had independent errors, a first-order
propagation would give

<p class="model-sky-equation">(<a class="model-sky-glossary-term" href="#model-sky-glossary-standard-deviation">σ</a><sub>Rp</sub>/R<sub>p</sub>)² ≈ (σ<sub>k</sub>/k)² + (σ<sub>R★</sub>/R<sub>★</sub>)².</p>

This illustrates the missing stellar-radius contribution; this notebook does
not perform that broader inference or assume these terms are independent in
a real stellar analysis. Changing limb darkening is a sensitivity experiment,
not a replacement for propagating uncertain assumptions jointly.

</details>

In [ ]:
# Change one assumption while preserving the measured light curve.
alternative_assumptions = {**assumptions, "limb_darkening": [0.5, 0.2]}
alternative_fit = pipeline.fit_transit(
    transit["times_minutes"], transit["flux"], transit["flux_error"],
    transit["valid"], alternative_assumptions)
print("Radius-ratio shift:",
      alternative_fit["radius_ratio"] - transit_fit["radius_ratio"])
print("Original conditional 1σ error:", transit_fit["radius_ratio_error"])

This shift comes from changing the star's assumed brightness pattern with the
measured data held fixed. A physical planet radius needs an <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a> budget
beyond the local optimizer's error bars.

<a id="model-sky-open-the-experiment-s-envelope"></a>

## Open the experiment's envelope

Write down your measured motion, <a class="model-sky-glossary-term" href="#model-sky-glossary-radius-ratio">radius ratio</a> and event midpoint before running
the next cell. <a class="model-sky-glossary-term" href="#model-sky-glossary-ground-truth">Injection</a> truth now becomes available for comparison. It was
not used by the measurement pipeline. A controlled experiment can expose a
biased answer even when its plots look convincing.

Compare like quantities. Geometric depth k² is a ratio of projected areas;
maximum dimming of a <a class="model-sky-glossary-term" href="#model-sky-glossary-limb-darkening">limb-darkened</a> star is different. The designed area ratio
is about 1.343%; peak dimming additionally depends on the adopted intensity
law, trajectory and <a class="model-sky-glossary-term" href="#model-sky-glossary-exposure-integration">exposure integration</a>. The motion target is about
94.8 mas/year across ten simulated years. Compare deviations with the stated
errors rather than rounding them away.


<details class="model-sky-science"><summary>Science Explanation</summary>

Existing qualification repeated the pipeline with five independent exposure-
<a class="model-sky-glossary-term" href="#model-sky-glossary-noise">noise</a> seeds on this fixed scene. All motion, radius-ratio and timing recoveries
fell within three reported errors. Five draws give only a rough check, not a
precise estimate of statistical <a class="model-sky-glossary-term" href="#model-sky-glossary-coverage">coverage</a> across arbitrary fields. They do not
sample different stellar populations or uncertain <a class="model-sky-glossary-term" href="#model-sky-glossary-orbit">orbital</a>, <a class="model-sky-glossary-term" href="#model-sky-glossary-spectrum">spectrum</a>,
limb-darkening and instrument parameters. The underlying analytic field has
no coadd-measurement error; this simplification is part of the qualification
boundary. These historical checks describe this Model Sky configuration;
results from other scenes are not substituted for them.

Negative controls ask whether the procedure reports a signal when it should
not: stationary targets, constant light, deliberately incompatible <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a>
coordinates and fully <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">masked</a> inputs. The contaminated <a class="model-sky-glossary-term" href="#model-sky-glossary-cadence">cadence</a> should be rejected. An intentionally wrong
<a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> tests model sensitivity; earlier qualification found biased <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a> and worse
<a class="model-sky-glossary-term" href="#model-sky-glossary-residual">residuals</a>. These are historical tests with their own package receipts, not
additional experiments silently performed by this truth-display cell. Fresh
execution establishes that this notebook runs; it does not expand those
scientific coverage claims.

</details>

In [ ]:
import json
# First access to injected answers: measurement and fitting are complete.
truth_path = pipeline.bundle_path(root, manifest["truth"]["json"])
truth = json.loads(truth_path.read_text())
display(Markdown("```json\n" + json.dumps(truth, indent=2) + "\n```"))

<a id="model-sky-extension-shorten-the-baseline-keep-the-star"></a>

### Extension · Shorten the baseline, keep the star

The optional cell rebuilds a separate controlled bundle with two years of
motion instead of ten. It reuses the Model Sky analytic <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">calibration</a> and
physical configuration, preserves the shared inputs, and writes a separate
bundle under `results/`. Predict whether the <a class="model-sky-glossary-term" href="#model-sky-glossary-dipole">dipole</a> still determines a
unique <a class="model-sky-glossary-term" href="#model-sky-glossary-displacement">displacement</a>. A shorter interval makes the <a class="model-sky-glossary-term" href="#model-sky-glossary-lobe">lobes</a> overlap more closely.

Run once in a fresh output directory. A message that motion is unconstrained
is informative for this case; it is not an instruction to relax the guard
until the intended answer appears.


<details class="model-sky-science"><summary>Science Explanation</summary>

For a small shift vector Δ, a difference between nearly overlapping <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSFs</a> is
approximately proportional to source <a class="model-sky-glossary-term" href="#model-sky-glossary-flux">flux</a> times Δ times the PSF <a class="model-sky-glossary-term" href="#model-sky-glossary-gradient">gradient</a>.
A brighter source moving less can resemble a fainter source moving more.
This is an <a class="model-sky-glossary-term" href="#model-sky-glossary-identifiability">identifiability</a> problem: the data constrain a combination of
parameters more strongly than each parameter alone.

The example rejects a fit when the absolute flux/separation correlation
exceeds 0.98. That conservative guard is not a universal telescope resolution
limit. A longer baseline or direct position fits in separate <a class="model-sky-glossary-term" href="#model-sky-glossary-epoch">epochs</a> can
supply different information. Even with a fixed position error, converting
to motion divides by elapsed time, so a shorter baseline increases the error
in mas/year.

</details>

In [ ]:
# Attendee opt-in; the executed reference enables this block.
RUN_BASELINE_EXERCISE = False
if RUN_BASELINE_EXERCISE:
    from dataclasses import replace
    from model_sky import SceneConfig, prepare_bundle
    # Separate generated dataset; refuse to overwrite an earlier experiment.
    variant_dir = OUTPUT / "baseline-2yr"
    if variant_dir.exists():
        raise FileExistsError("Choose a new output for this experiment")
    config = replace(SceneConfig(), baseline_years=2.0)
    prepare_bundle(root / manifest["calibration_dir"], variant_dir,
                   config=config)
    variant_root, variant_manifest = pipeline.load_bundle(variant_dir)
    variant_files = variant_manifest["files"]
    earlier = pipeline.read_exposure(pipeline.bundle_path(
        variant_root, variant_files["reference_old"]))
    later = pipeline.read_exposure(pipeline.bundle_path(
        variant_root, variant_files["motion"]))
    try:
        # Rejection can be correct when flux and separation are degenerate.
        recovered = pipeline.measure_motion(
            earlier, later, variant_manifest["target_search"]["old_xy"])
    except RuntimeError as exc:
        print("Motion is unconstrained for this experiment:", exc)
    else:
        factor = 1000 * variant_manifest["pixel_scale_arcsec"] / 2
        print("Two-year motion [mas/year]:",
              recovered["separation_pixels"] * factor,
              "+/-", recovered["separation_error_pixels"] * factor)

For the prepared two-year case, the flux/separation guard reports that the
motion is unconstrained. That outcome is part of the experiment: reducing
the baseline removes information even though the same GPU fitting routine
still runs. Preserve the rejection alongside the ten-year result instead of
reporting the unstable fit as a second precise measurement.

<a id="model-sky-keep-the-result-inspectable"></a>

## Keep the result inspectable

Save arrays, summaries and the three interactive chapter views. Their HTML
embeds plots and displayed data for offline inspection. Moving a slider selects
existing measurements; changing a scientific assumption still requires
rerunning the corresponding Python or CLI step.

Chapter files and intermediates live under `results/`. The runner saves
`model-sky.executed.ipynb` alongside that directory and records input and source
hashes in `execution.json`. Its output directory contains the source, prepared
inputs and measured results needed to inspect that execution. The source
notebook remains free of stored outputs.

In [ ]:
workshop.export_views(OUTPUT, manifest, trail, motion, transit)
print("Saved measurement arrays and standalone HTML to", OUTPUT)

<details><summary>Run the prepared experiment from a terminal</summary>

```bash
python workshop.py all --bundle data --output results
```

This example runner connects <a class="model-sky-glossary-term" href="#model-sky-glossary-registration">registration</a>, <a class="model-sky-glossary-term" href="#model-sky-glossary-mask">masks</a>, <a class="model-sky-glossary-term" href="#model-sky-glossary-photometry">photometry</a> and the
<a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a> model. Product commands serve individual stages; this runner
is an example workflow, not an additional cuPhoton product command.

</details>

<a id="model-sky-optional-investigation-could-another-system-make-this-dip"></a>

## Optional investigation · Could another system make this dip?

A dip alone does not establish a planet. An <a class="model-sky-glossary-term" href="#model-sky-glossary-blend">unresolved</a> **blend** combines light
from multiple sources in the same footprint. Two stars <a class="model-sky-glossary-term" href="#model-sky-glossary-eclipse">eclipsing</a> each other
can lose a great deal of their own light; a brighter neighbor dilutes that
loss into a shallow dip that resembles a planetary <a class="model-sky-glossary-term" href="#model-sky-glossary-transit">transit</a>.

The optional paired experiment uses the same Model Sky <a class="model-sky-glossary-term" href="#model-sky-glossary-calibration">calibration</a> and
physical star model to compare a planet on the target with a fading neighbor.
The nearby contaminant is added for this experiment; it is not an observed
companion or a catalog entry. Each case supplies out-of-event and in-event images, independent
16-exposure averages at those light levels rather than complete time series.
xFit fits centers of the source light profiles, a model-based <a class="model-sky-glossary-term" href="#model-sky-glossary-centroid">centroid</a>
measurement, and locates the negative difference signal. A similar <a class="model-sky-glossary-term" href="#model-sky-glossary-aperture">aperture</a> depth can hide a
different location for missing light. An aperture is a region whose pixels
are summed.

Inspect the cyan reference-position marker and measured fading-light position.
These controlled snapshots demonstrate source localization. They cannot reveal
alternating eclipse depths or a second eclipse that was never observed, and a
small offset does not prove a planet. Repeated events, additional colors,
spectra and longer time <a class="model-sky-glossary-term" href="#model-sky-glossary-coverage">coverage</a> provide different tests.


<details class="model-sky-science"><summary>Science Explanation</summary>

Here D = I<sub>in</sub> − I<sub>out</sub>, so the fitted fading source has
negative amplitude. Aperture depth is 1 − F<sub>in</sub>/F<sub>out</sub>.
Extra constant neighbor light contributes to both sums and reduces fractional
depth. The <a class="model-sky-glossary-term" href="#model-sky-glossary-difference">difference image</a> can instead locate the changing light.

Formal centroid errors condition on a chosen <a class="model-sky-glossary-term" href="#model-sky-glossary-gaussian">Gaussian</a> model and estimated sky.
The difference-source and <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> centroid share pixels, so their separation
has no independent significance claim. This does not propagate every <a class="model-sky-glossary-term" href="#model-sky-glossary-psf">PSF</a> or
background-model <a class="model-sky-glossary-term" href="#model-sky-glossary-uncertainty">uncertainty</a>. [Bryson et al. (2013)](https://arxiv.org/abs/1303.0052)
describe centroid vetting for Kepler <a class="model-sky-glossary-term" href="#model-sky-glossary-false-positive">false positives</a>.

Transit <a class="model-sky-glossary-term" href="#model-sky-glossary-photometry">photometry</a> measures light loss. Other planet searches measure different
effects: <a class="model-sky-glossary-term" href="#model-sky-glossary-radial-velocity">radial velocity</a> detects a star's motion toward and away from us;
<a class="model-sky-glossary-term" href="#model-sky-glossary-astrometry">astrometry</a> detects its wobble across the sky; <a class="model-sky-glossary-term" href="#model-sky-glossary-microlensing">microlensing</a> detects gravitational
magnification; <a class="model-sky-glossary-term" href="#model-sky-glossary-direct-imaging">direct imaging</a> separates companion light. Combining a mass
constraint with radius can constrain density. See
[NASA's detection-method overview](https://science.nasa.gov/exoplanets/how-we-find-and-characterize/)
and [AAVSO follow-up photometry](https://aavso.org/exoplanet-photometry/).

</details>

In [ ]:
# Independent in/out snapshots locate missing light in two cases.
RUN_BLEND_EXTENSION = False
if RUN_BLEND_EXTENSION:
    import blend
    from model_sky import SceneConfig
    blend_result = blend.run(root, OUTPUT / "blend", config=SceneConfig(), sources=Path("sources.md"))
    show_dark(blend_result["layout"])
    print("Saved blend diagnostics to", blend_result["html"])

Compare the location of missing light with the target's <a class="model-sky-glossary-term" href="#model-sky-glossary-reference">reference</a> position.
Similar total dimming can have different spatial explanations. These paired
images support a localization lesson; they do not establish a planet or
supply an independent significance for every <a class="model-sky-glossary-term" href="#model-sky-glossary-centroid">centroid</a> offset.

xScan is omitted from Model Sky. No trained model, classifier performance or
inference is claimed for these images.

<a id="model-sky-further-reading"></a>

## Further reading

The bundled source list records scientific techniques, data attribution and
model inputs. The text and glossary explain terms locally; these links support
deeper reading when you have a network connection. Opening external references
is optional for running or viewing the example.

In [ ]:
display(Markdown(Path("sources.md").read_text(encoding="utf-8")))

<a id="model-sky-science-glossary"></a>

## Science glossary

Dotted terms show a definition on hover or keyboard focus in the HTML. Click or tap one to jump to its entry here; your browser's Back button returns to the reading position. In a notebook, the same links and native hover text work without an extension.

<dl class="model-sky-glossary" aria-label="Science terms in Model Sky">
<dt id="model-sky-glossary-quadrature" tabindex="-1">Adding errors in quadrature</dt>
<dd>Squaring independent uncertainty sizes, adding those squares, then taking the square root. This combines independent variance contributions; correlations require additional covariance terms.</dd>
<dt id="model-sky-glossary-angular-scale" tabindex="-1">Angular scale / pixel scale</dt>
<dd>How much angle on the sky corresponds to one image pixel. The model declares 0.20 arcseconds per pixel. Multiplying pixel separation by this scale gives an angle; converting that angle to a physical distance also requires the distance to the star.</dd>
<dt id="model-sky-glossary-aperture" tabindex="-1">Aperture / annulus</dt>
<dd>For photometry, an aperture is a selected region around a source; an annulus is a surrounding ring often used to estimate local sky. Aperture can also mean the telescope&#x27;s light-collecting opening.</dd>
<dt id="model-sky-glossary-arcsecond" tabindex="-1">Arcsecond / milliarcsecond</dt>
<dd>An arcsecond is 1/3,600 of a degree, a unit of angle. A milliarcsecond (mas) is 1/1,000 of an arcsecond. mas/yr measures angular change per year; neither unit is a time interval despite the word second.</dd>
<dt id="model-sky-glossary-astrometry" tabindex="-1">Astrometry</dt>
<dd>Measuring where objects appear on the sky and how those positions change. Image alignment and the moving star&#x27;s position measurement are astrometric tasks.</dd>
<dt id="model-sky-glossary-background" tabindex="-1">Background / sky level</dt>
<dd>Light or detector signal surrounding the source being measured. A local sky estimate removes a nearby baseline level. The modeled background scene contains fixed fictional stars. Diffuse sky light contributes noise even though its mean level has been subtracted.</dd>
<dt id="model-sky-glossary-bandpass" tabindex="-1">Bandpass / modeled band</dt>
<dd>The range of light wavelengths admitted by an instrument, weighted by its sensitivity. Model Sky uses a declared response from 560 to 700 nanometres. This interval is an idealized model, not a measured filter curve or a standard astronomical band.</dd>
<dt id="model-sky-glossary-blackbody" tabindex="-1">Blackbody / Planck spectrum</dt>
<dd>An ideal thermal emitter whose smooth spectrum is set by its temperature through Planck&#x27;s law. Here 3100 K, an assumed stellar radius and distance set the received light. Real cool stars have molecular absorption missing from this approximation; it is not a measured stellar spectrum.</dd>
<dt id="model-sky-glossary-blend" tabindex="-1">Blend / unresolved source</dt>
<dd>Light from different objects overlaps too closely for the image to separate them. A faint eclipsing star behind the target can produce a dip attributed to the wrong object. Unresolved means the components are not separately distinguished.</dd>
<dt id="model-sky-glossary-cadence" tabindex="-1">Cadence</dt>
<dd>The timing or spacing of observations. Here one 30-second exposure starts each minute. A cadence can also mean one sample in that sequence; rejecting a cadence excludes that exposure from the measurement.</dd>
<dt id="model-sky-glossary-calibration" tabindex="-1">Calibration / calibrated flux</dt>
<dd>Putting measurements on a common physical or positional scale. Here the electron-to-nJy conversion, pixel scale and coordinate grid are declared model inputs. They keep units consistent within the experiment and do not represent calibration measurements from an actual camera.</dd>
<dt id="model-sky-glossary-centroid" tabindex="-1">Centroid</dt>
<dd>The estimated centre of an object&#x27;s light distribution. Fitting several pixels can locate it to a fraction of a pixel. A changing centroid can reveal a blend or an alignment problem.</dd>
<dt id="model-sky-glossary-check-star" tabindex="-1">Check star</dt>
<dd>A separate star treated as steady and measured like the target, but kept out of the brightness correction. Its remaining fluctuations check whether quoted errors describe the scatter in this field.</dd>
<dt id="model-sky-glossary-chi-square" tabindex="-1">Chi-square / reduced chi-square</dt>
<dd>A sum of squared data-minus-model differences divided by expected variances. Reduced chi-square divides by independent constraints left after fitting. Values above roughly one can signal underestimated errors or a poor model; correlations complicate that rule.</dd>
<dt id="model-sky-glossary-coadd" tabindex="-1">Coadd / averaged reference</dt>
<dd>An image combining aligned exposures, often by averaging. Each Model Sky reference is generated with the noise of a mean of 16 independent, equal-duration exposures. It still has noise shared by later measurements that reuse it. The underlying model sky itself is noiseless.</dd>
<dt id="model-sky-glossary-comparison-star" tabindex="-1">Comparison star / ensemble</dt>
<dd>A nearby star assumed steady during the sequence. Changes common to several such stars estimate changes in observing conditions. An ensemble combines their measurements to correct the target&#x27;s light curve.</dd>
<dt id="model-sky-glossary-conditional" tabindex="-1">Conditional uncertainty</dt>
<dd>An error estimate holding stated assumptions fixed. Here the analytic sky, stellar properties, orbit and much of the detector model are fixed while exposure noise varies. Repeated noise realizations do not cover every way those model assumptions could be wrong.</dd>
<dt id="model-sky-glossary-convolution" tabindex="-1">Convolution</dt>
<dd>Replacing each pixel with a weighted combination of nearby pixels. The weights form a small array called a kernel. Here convolution broadens reference stars to match the science image&#x27;s blur.</dd>
<dt id="model-sky-glossary-correlated-noise" tabindex="-1">Correlated noise</dt>
<dd>Errors that move together. Resampling can reuse one input pixel in several output pixels, correlating their errors. A shared reference can also correlate measurements from different exposures.</dd>
<dt id="model-sky-glossary-cosine" tabindex="-1">Cosine / surface normal</dt>
<dd>Cosine describes the projection of one direction onto another: 1 when aligned and 0 at a right angle. A surface normal is perpendicular to a surface. In limb darkening, the angle describes how directly we view a patch of the star.</dd>
<dt id="model-sky-glossary-covariance" tabindex="-1">Covariance</dt>
<dd>A measure of how errors in two quantities vary together. A covariance matrix records these relationships as well as each quantity&#x27;s variance. It matters when combining fitted positions into a separation error.</dd>
<dt id="model-sky-glossary-degrees-of-freedom" tabindex="-1">Degrees of freedom</dt>
<dd>Independent constraints remaining after fitting parameters. For a simple fit with independent errors, this is the number of data points minus the number of fitted parameters.</dd>
<dt id="model-sky-glossary-difference" tabindex="-1">Difference image / difference flux</dt>
<dd>Science minus the PSF-matched reference and fitted background. Positive pixels mean extra light relative to that reference; negative pixels mean less. Difference flux measures the change, not all the light from a source.</dd>
<dt id="model-sky-glossary-dipole" tabindex="-1">Dipole</dt>
<dd>A paired positive and negative patch in a difference image. When a star moves, its new position leaves added light and its old position leaves missing light after subtraction. Dipole means two lobes here, not an electrical property of the star. Misaligned images can make similar patterns.</dd>
<dt id="model-sky-glossary-direct-imaging" tabindex="-1">Direct imaging</dt>
<dd>Separating and detecting light from a planet beside its much brighter host star. This differs from inferring a planet through a change in the star&#x27;s total light.</dd>
<dt id="model-sky-glossary-displacement" tabindex="-1">Displacement</dt>
<dd>A change in position with both size and direction. New position minus old position gives a two-component displacement in pixels. Its length is the separation.</dd>
<dt id="model-sky-glossary-eccentricity" tabindex="-1">Eccentricity</dt>
<dd>How stretched an orbit is: zero is a circle, while larger values below one describe increasingly elongated bound orbits. The adopted shape affects how fast a planet crosses the star.</dd>
<dt id="model-sky-glossary-eclipse" tabindex="-1">Eclipse / eclipsing binary</dt>
<dd>Dimming when one body passes in front of another. An eclipsing binary is a pair of orbiting stars whose alignment causes dips. Its light can mimic a planet transit, especially when mixed with another star&#x27;s light.</dd>
<dt id="model-sky-glossary-electron" tabindex="-1">Electron / detector count</dt>
<dd>A particle carrying electric charge. A camera records charge produced when light is detected. Here electron counts are simulated using the declared telescope area, exposure time and efficiency, then converted to nJy. Photon-counting fluctuations and read noise enter in these count units; no raw camera counts are imported.</dd>
<dt id="model-sky-glossary-empirical" tabindex="-1">Empirical</dt>
<dd>Estimated from measurements rather than supplied as a model parameter. Here the measurement pipeline estimates PSFs from stars in the simulated images and error scales from their measured scatter. These estimates are independent of the withheld injection truth; empirical does not mean an observed astronomical image was imported.</dd>
<dt id="model-sky-glossary-epoch" tabindex="-1">Epoch</dt>
<dd>The time associated with an observation or reference position. The old and current motion epochs are simulated points ten years apart; the short transit sequence samples the current epoch.</dd>
<dt id="model-sky-glossary-exoplanet" tabindex="-1">Exoplanet</dt>
<dd>A planet outside our Solar System. A transit-shaped dip is evidence to investigate, not sufficient proof that an exoplanet caused it.</dd>
<dt id="model-sky-glossary-exposure" tabindex="-1">Exposure / visit</dt>
<dd>A period during which a detector collects light, and the resulting measurement. A visit is a telescope observation with timing and pointing. Each simulated exposure here averages incoming light over 30 seconds.</dd>
<dt id="model-sky-glossary-exposure-integration" tabindex="-1">Exposure integration</dt>
<dd>Averaging the model over the time the detector collects light. A 30-second exposure does not measure a single instant; averaging can soften the transit&#x27;s entry and exit compared with an instantaneous model.</dd>
<dt id="model-sky-glossary-extinction" tabindex="-1">Extinction</dt>
<dd>Dimming of starlight by absorption or scattering along its path, for example by interstellar dust. The Model Sky brightness calculation sets interstellar extinction to zero. That assumption is distinct from the instrument&#x27;s modeled photon-detection efficiency.</dd>
<dt id="model-sky-glossary-false-positive" tabindex="-1">False positive</dt>
<dd>A signal accepted as the event of interest even though another cause produced it. Blended eclipsing stars and instrumental changes can produce planet-like brightness dips.</dd>
<dt id="model-sky-glossary-fits" tabindex="-1">FITS / HDU / plane</dt>
<dd>FITS is an astronomy file format for numerical arrays and metadata. Each header/data unit (HDU) is one named or numbered section. A plane here is a two-dimensional array: brightness, estimated variance, or pixel-quality flags on matching coordinates.</dd>
<dt id="model-sky-glossary-flux" tabindex="-1">Flux / flux density</dt>
<dd>A measure of received light. This example uses nJy, a flux-density unit describing brightness per unit frequency. Summing or fitting a star&#x27;s pixel pattern estimates its total brightness in that band.</dd>
<dt id="model-sky-glossary-forced-photometry" tabindex="-1">Forced PSF photometry</dt>
<dd>Fitting a known star-shaped light pattern at a supplied position to measure brightness. Forced means the position is supplied even if that exposure alone has too little signal to detect the source.</dd>
<dt id="model-sky-glossary-gaussian" tabindex="-1">Gaussian</dt>
<dd>A smooth bell-shaped function used to approximate blur or random-error distributions. A Gaussian-polynomial basis combines bell shapes with coordinate-dependent terms to describe more varied kernels.</dd>
<dt id="model-sky-glossary-gradient" tabindex="-1">Gradient / propagated error</dt>
<dd>A gradient records how a result changes for a small change in each input. Error propagation uses those sensitivities and the input covariance to estimate the result&#x27;s uncertainty. This approximation is local to the fitted values.</dd>
<dt id="model-sky-glossary-microlensing" tabindex="-1">Gravitational microlensing</dt>
<dd>Temporary brightening when a foreground object&#x27;s gravity bends and magnifies light from a more distant star. A planet can alter that pattern, providing a detection method distinct from transits.</dd>
<dt id="model-sky-glossary-ground-truth" tabindex="-1">Ground truth / injection</dt>
<dd>Known values used to create a simulated signal, such as position or planet radius. An injection adds that controlled signal to the scene. Truth is withheld from measurement so recovery can be checked afterward.</dd>
<dt id="model-sky-glossary-identifiability" tabindex="-1">Identifiability / degeneracy</dt>
<dd>Whether data can distinguish parameter choices. A brighter star with smaller motion can produce nearly the same pattern as a fainter star with larger motion; their values are then degenerate. A converged fit does not resolve this ambiguity by itself.</dd>
<dt id="model-sky-glossary-kernel" tabindex="-1">Image-matching kernel / basis</dt>
<dd>A small array of convolution weights reshaping the reference to match the science image. The basis supplies allowed patterns whose coefficients are fitted. This differs from a GPU program kernel or a notebook&#x27;s Python process.</dd>
<dt id="model-sky-glossary-impact-parameter" tabindex="-1">Impact parameter</dt>
<dd>How far from the star&#x27;s apparent centre the planet passes at mid-transit, measured in stellar radii. Zero crosses the centre; a value near one passes near the edge and changes the shape and duration.</dd>
<dt id="model-sky-glossary-interpolation" tabindex="-1">Interpolation / resampling</dt>
<dd>Estimating values between existing samples. Here, reprojection resamples input pixels onto a new grid using weighted neighbours. This changes how noise is shared between pixels; it does not create new independent data.</dd>
<dt id="model-sky-glossary-latent-sky" tabindex="-1">Latent sky / noiseless scene</dt>
<dd>The underlying expected image before random measurement noise is drawn. Here it is built from fictional stars and analytic light profiles. Each exposure applies its own blur, shift and brightness changes before receiving independent noise. Reusing this noiseless model does not reuse a previous exposure&#x27;s random errors.</dd>
<dt id="model-sky-glossary-least-squares" tabindex="-1">Least-squares fit</dt>
<dd>Choosing parameters to minimize squared data-minus-model differences. Weighting gives more reliable measurements more influence, often by dividing each squared difference by its estimated variance.</dd>
<dt id="model-sky-glossary-light-curve" tabindex="-1">Light curve</dt>
<dd>Brightness plotted against time. Here each point is one exposure&#x27;s brightness after correcting changes shared with comparison stars. A transit appears as a temporary downward dip.</dd>
<dt id="model-sky-glossary-light-year" tabindex="-1">Light-year</dt>
<dd>The distance light travels in one year in vacuum, about 9.46 trillion kilometres. It is a distance, not a duration.</dd>
<dt id="model-sky-glossary-limb-darkening" tabindex="-1">Limb darkening</dt>
<dd>The star&#x27;s visible edge is dimmer than its centre, so a crossing planet blocks different amounts of light along its path. This experiment declares a quadratic intensity law with coefficients 0.6 and 0.15. That shape is prescribed separately from the blackbody calculation of the star&#x27;s total brightness.</dd>
<dt id="model-sky-glossary-magnitude" tabindex="-1">Magnitude / AB magnitude</dt>
<dd>Astronomy&#x27;s logarithmic brightness scale: smaller numbers mean brighter objects, and five magnitudes correspond to a factor of 100 in brightness. The AB convention uses a reference flux density of 3631 Jy. The modeled star has AB magnitude about 17.54 through the declared bandpass.</dd>
<dt id="model-sky-glossary-marginal" tabindex="-1">Marginal variance / uncertainty</dt>
<dd>Here marginal variance means one pixel&#x27;s individual variance, a diagonal entry of the joint covariance. It does not include cross-pixel covariance or imply that kernel-fit uncertainty was integrated out. In other contexts marginalizing parameters means averaging over their possible values.</dd>
<dt id="model-sky-glossary-mask" tabindex="-1">Mask</dt>
<dd>Flags or Boolean selections describing which measurements a stage may use. A quality mask records defects; a fit mask selects allowed pixels. Check the convention: True can mean included for a fit mask and excluded for a trail mask. Masking discards information.</dd>
<dt id="model-sky-glossary-mid-transit" tabindex="-1">Mid-transit time</dt>
<dd>The time at the centre of the transit. The fit estimates when the crossing is centred relative to the sequence&#x27;s time origin. This differs from measuring the orbital period between repeated transits.</dd>
<dt id="model-sky-glossary-nan" tabindex="-1">NaN / invalid measurement</dt>
<dd>A missing or invalid numerical result. A rejected exposure receives NaN flux rather than zero: zero would be a real brightness measurement and could falsely imply that the star disappeared.</dd>
<dt id="model-sky-glossary-njy" tabindex="-1">nJy (nanojansky)</dt>
<dd>One nanojansky is 10⁻⁹ jansky, or 10⁻³⁵ watts per square metre per hertz. Here each pixel contains its contribution to modeled flux density. Summing or fitting a star&#x27;s pattern estimates its brightness in the declared band. Variance in nJy² describes squared brightness uncertainty.</dd>
<dt id="model-sky-glossary-noise" tabindex="-1">Noise / photon noise</dt>
<dd>Random variation in a measurement. Photon arrivals fluctuate even for a steady star; sky light and the detector add variation too. Reusing one noisy image copies its errors instead of generating independent exposures.</dd>
<dt id="model-sky-glossary-normalization" tabindex="-1">Normalization / relative flux</dt>
<dd>Rescaling to a chosen reference. A normalized PSF has pixel weights adding to 1. A light curve divided by its usual brightness is near 1; 0.99 is one per cent dimmer. Comparison-star normalization corrects changes shared across the field.</dd>
<dt id="model-sky-glossary-orbit" tabindex="-1">Orbit / orbital period</dt>
<dd>The path of one body around another under gravity. The orbital period is the time for a full circuit. The example assumes a period from its chosen model; one short transit sequence does not measure it.</dd>
<dt id="model-sky-glossary-inclination" tabindex="-1">Orbital inclination</dt>
<dd>The tilt of the orbit relative to the sky plane. In the transit model, 90 degrees means the orbit is viewed edge-on, making a crossing possible.</dd>
<dt id="model-sky-glossary-parallax" tabindex="-1">Parallax</dt>
<dd>An apparent position shift caused by a changing viewpoint. Annual parallax follows Earth&#x27;s orbit and helps measure distance. This experiment models a star&#x27;s linear proper motion only; it includes no annual parallax signal and does not measure the assumed stellar distance.</dd>
<dt id="model-sky-glossary-parsec" tabindex="-1">Parsec (pc)</dt>
<dd>A distance unit of about 3.26 light-years. One parsec corresponds to a star with annual parallax of one arcsecond. The simulated star&#x27;s 146.42 pc distance is an assumption, not a distance measured here.</dd>
<dt id="model-sky-glossary-photometry" tabindex="-1">Photometry</dt>
<dd>Measuring an object&#x27;s brightness. This notebook fits the star&#x27;s expected pattern to pixels and repeats the measurement through time to obtain a light curve.</dd>
<dt id="model-sky-glossary-photon" tabindex="-1">Photon</dt>
<dd>A quantum, or discrete packet, of light. A detector converts incoming photons into an electrical signal. Photon arrival counts fluctuate, so even a perfectly steady source has measurement noise.</dd>
<dt id="model-sky-glossary-poisson" tabindex="-1">Poisson statistics</dt>
<dd>A model for independent random event counts. For a mean count N, count variance is N and standard deviation is √N. Calibrated flux units and detector/background effects must be included before applying this rule to image values.</dd>
<dt id="model-sky-glossary-lobe" tabindex="-1">Positive / negative lobe</dt>
<dd>One patch of a dipole&#x27;s two-part pattern. The positive lobe has more light in science than reference; the negative lobe has less. Negative values describe subtraction, not physically negative light.</dd>
<dt id="model-sky-glossary-proper-motion" tabindex="-1">Proper motion</dt>
<dd>A star&#x27;s gradual angular movement across the sky, usually in milliarcseconds per year. It is not the full three-dimensional speed; converting to a sideways physical speed also requires distance.</dd>
<dt id="model-sky-glossary-psf" tabindex="-1">PSF (point-spread function)</dt>
<dd>The pixel pattern made by one unresolved point of light. Preparation uses a Gaussian profile plus each exposure&#x27;s common seeing blur for both field and target stars. Measurement estimates a PSF from image stars without reading that input profile. Matching PSFs makes images comparable before subtraction.</dd>
<dt id="model-sky-glossary-radial-velocity" tabindex="-1">Radial velocity</dt>
<dd>Speed toward or away from the observer. A planet and star orbit their common centre of mass; changes in the star&#x27;s radial velocity can reveal the planet. This needs spectral observations, not just the transit light curve.</dd>
<dt id="model-sky-glossary-radius-ratio" tabindex="-1">Radius ratio / transit depth</dt>
<dd>Radius ratio k is planet radius divided by star radius. For a small planet crossing a uniformly bright star, fractional blocked light is about k². Limb darkening and the crossing path change the actual dip. Physical planet radius also needs an assumed stellar radius.</dd>
<dt id="model-sky-glossary-read-noise" tabindex="-1">Read noise</dt>
<dd>Random variation introduced when detector electronics read accumulated charge. It can occur even with no incoming light. The model assigns 5 electrons of standard deviation per pixel per exposure and combines its variance with sky noise before converting to nJy².</dd>
<dt id="model-sky-glossary-reference" tabindex="-1">Reference / template</dt>
<dd>The comparison image against which science is measured. An old-position reference reveals motion. A current-position, outside-transit reference measures dimming at the present location. The reference also has noise.</dd>
<dt id="model-sky-glossary-registration" tabindex="-1">Registration</dt>
<dd>Aligning images so the same array coordinates describe the same sky position. WCS metadata gives the initial mapping; stationary neighbouring stars check the remaining alignment error.</dd>
<dt id="model-sky-glossary-residual" tabindex="-1">Residual</dt>
<dd>What remains after subtracting a model from data. In image subtraction it highlights changes and mismatches. After a dipole or transit fit, structured leftovers can show that the model missed something.</dd>
<dt id="model-sky-glossary-rms" tabindex="-1">RMS (root mean square)</dt>
<dd>Square each value, average the squares, then take the square root. For residuals this summarizes typical size in original units without positive and negative values cancelling. Large outliers strongly affect it.</dd>
<dt id="model-sky-glossary-sampling" tabindex="-1">Sampling / undersampling</dt>
<dd>Representing a continuous pattern with finite pixels or time steps. Undersampling means too few samples to resolve its shape. Subpixel describes positions or model steps finer than one detector pixel; it does not add new observed pixels.</dd>
<dt id="model-sky-glossary-saturation" tabindex="-1">Saturation</dt>
<dd>A detector or recorded value reaching its usable upper limit. Such a pixel no longer reliably measures how much light arrived. Model Sky does not simulate saturation; real-camera data would need suitable quality flags and detector-specific limits.</dd>
<dt id="model-sky-glossary-science-image" tabindex="-1">Science image</dt>
<dd>The exposure being investigated, compared with a reference image. Registered science is that measurement resampled onto the reference grid. Science here labels the image&#x27;s role in the comparison.</dd>
<dt id="model-sky-glossary-seeing" tabindex="-1">Seeing</dt>
<dd>Blurring and distortion caused by Earth&#x27;s turbulent atmosphere. Here a declared blur varies across exposures and affects all stars in each exposure consistently. It illustrates changing image quality without importing an atmospheric measurement or simulating all turbulence effects.</dd>
<dt id="model-sky-glossary-signal-to-noise" tabindex="-1">Signal-to-noise ratio (S/N)</dt>
<dd>Signal size compared with estimated random uncertainty. Higher values make random fluctuations a less plausible explanation, but do not rule out systematic errors or a wrong model.</dd>
<dt id="model-sky-glossary-sky-plane" tabindex="-1">Sky plane / line of sight</dt>
<dd>The sky plane describes directions across the image, perpendicular to the direction toward the observer. Motion in this plane changes apparent position; line-of-sight motion is toward or away from us.</dd>
<dt id="model-sky-glossary-spectrum" tabindex="-1">Spectrum / stellar atmosphere</dt>
<dd>A spectrum describes light at each wavelength or frequency. A stellar atmosphere is the outer gas that shapes a star&#x27;s escaping light. Model Sky uses a smooth blackbody spectrum in place of an atmosphere model; molecular absorption and other spectral features are omitted.</dd>
<dt id="model-sky-glossary-stamp" tabindex="-1">Stamp</dt>
<dd>A small image cutout around an object or candidate. Fitting a stamp uses nearby pixels without processing the full field for each source.</dd>
<dt id="model-sky-glossary-standard-deviation" tabindex="-1">Standard deviation / sigma</dt>
<dd>A measure of spread, equal to the square root of variance. For an accurate Gaussian error model about 68% of repeated errors fall within ±1 sigma. That interpretation needs the model assumptions to hold.</dd>
<dt id="model-sky-glossary-stellar-radius" tabindex="-1">Stellar radius / solar radius</dt>
<dd>The distance from a star&#x27;s centre to its visible surface. A solar radius uses the Sun&#x27;s radius as a unit. Here an assumed stellar radius turns the fitted planet-to-star radius ratio into a planet size.</dd>
<dt id="model-sky-glossary-sum" tabindex="-1">Sum / summation</dt>
<dd>Adding a collection of numbers. The symbol Σ means sum over the stated index. A weighted sum first multiplies each value by its weight; a weighted average divides by the total weight.</dd>
<dt id="model-sky-glossary-systematics" tabindex="-1">Systematic errors / systematics</dt>
<dd>Repeatable biases or patterns left by calibration, observing conditions or a wrong model. More samples may not average them away. Comparison stars and residuals help expose some of them.</dd>
<dt id="model-sky-glossary-throughput" tabindex="-1">Throughput / instrument response</dt>
<dd>The fraction of incoming light converted into recorded signal after all optical and detector losses. Here the declared total photon efficiency is 0.4 from 560 to 700 nanometres and zero outside that band. This is an optical model input, not GPU operations per second.</dd>
<dt id="model-sky-glossary-baseline" tabindex="-1">Time baseline</dt>
<dd>Elapsed time between position measurements. A longer interval lets a slowly moving star travel farther in the image. The default ten-year interval is an input to the simulated motion experiment; it is not elapsed observing time or the length of the short transit sequence.</dd>
<dt id="model-sky-glossary-top-hat" tabindex="-1">Top-hat response</dt>
<dd>An idealized response that is constant inside a specified interval and zero outside it, with sharp edges. The Model Sky passband has this shape from 560 to 700 nanometres. Real filters generally have rounded edges and wavelength-dependent sensitivity.</dd>
<dt id="model-sky-glossary-transit" tabindex="-1">Transit / ingress / egress</dt>
<dd>A planet or other body crossing in front of its star as seen by us, temporarily blocking starlight. Ingress is entry onto the visible disc; egress is exit. The example creates a simulated planet transit with a physical light-curve model.</dd>
<dt id="model-sky-glossary-uncertainty" tabindex="-1">Uncertainty / formal error</dt>
<dd>An estimate of precision under stated assumptions. Formal fit error comes from the fit model and supplied noise. It can omit alignment errors, shared-reference errors or wrong assumptions; it is not a guaranteed bound.</dd>
<dt id="model-sky-glossary-coverage" tabindex="-1">Uncertainty coverage</dt>
<dd>For uncertainty checks, the fraction of repeated experiments whose error interval contains the true value. A few successful trials cannot establish that fraction precisely. Image coverage means which sky area or pixels are available.</dd>
<dt id="model-sky-glossary-variance" tabindex="-1">Variance</dt>
<dd>Expected squared fluctuation around a mean. Its square root is the standard deviation. Brightness in nJy has variance in nJy². A per-pixel variance plane records individual error sizes but not all correlations between pixels.</dd>
<dt id="model-sky-glossary-wavelength" tabindex="-1">Wavelength / frequency</dt>
<dd>Wavelength is the spacing between successive crests of a light wave. Frequency counts its oscillations per second. Higher frequency means shorter wavelength; visible colours correspond to different wavelengths.</dd>
<dt id="model-sky-glossary-wcs" tabindex="-1">WCS (World Coordinate System)</dt>
<dd>Metadata mapping pixels to sky coordinates. Right ascension and declination locate a direction, like longitude and latitude on Earth. Here the grid and exposure shifts are declared inputs around RA 210 degrees and Dec −10 degrees, not a solution fitted to a catalog. Stationary stars still check the image-registration accuracy.</dd>
<dt id="model-sky-glossary-reprojection" tabindex="-1">Reprojection</dt>
<dd>Resampling an image onto a specified sky coordinate grid using a supplied coordinate mapping. It does not itself estimate alignment corrections from stars.</dd>
</dl>

These definitions describe the Model Sky experiment. See the [NASA FITS primer](https://fits.gsfc.nasa.gov/fits_primer.html), [Astropy WCS documentation](https://docs.astropy.org/en/stable/wcs/wcsapi.html), [Astropy blackbody model](https://docs.astropy.org/en/stable/modeling/physical_models.html#blackbody), [NASA planet-detection overview](https://science.nasa.gov/exoplanets/how-we-find-and-characterize/) and the primary papers linked beside each Science Explanation.